# Haru AIO Downloader

Satu notebook untuk semua downloader & tools:

- **YouTube / Playlist** → `haru-ytdl` \(auto-upload **Gofile** + kirim link ke bot, opsional cookies.txt\)
- **LRC Lirik** → `haru-lrc` \(cari & download LRC, konversi ke SRT, kirim file ke bot\)
- **Manga 4 Sumber** → `haru-manga` (MangaDex, HentaiRead.com, Kanzenin.info, CrotPedia.net; download chapter, preview 10 halaman + ZIP otomatis dikirim ke bot)
- **Netflix Checker** → `haru-check` \(cek akun valid/hold/invalid, hasil + `Hits.zip` + ringkasan Telegra.ph ke bot\)
- **Transfer.it Renew** → `haru-transferit` \(perpanjang & hidupkan kembali semua transfer ke 90 hari\)
- **Subtitle SubSource** → `haru-sub` \(cari subtitle, lihat uploader/deskripsi/preview seperti di web, download & kirim ke bot\)


**Cara pakai:** jalankan cell **Install**, buka **Web Terminal**, ketik salah satu command di atas. Hasil masuk `/content/downloads/` per jenis.

> Companion dari `mkvtoolnix.ipynb` (muxing & extract MKV). Secrets dipakai bersama (`/content/.haru_secrets.json`).

---

### Persiapan

Buka menu **Rahasia** (ikon kunci), tambah secret berikut (aktifkan toggle-nya):

| Key | Value | Keterangan |
|-----|-------|------------|
| `HARU_BOT_TOKEN` | *(token BotFather)* | Notif & kirim file Telegram |
| `OWNER_ID` | *(chat ID)* | Tujuan notif Telegram |
| `GOFILE_API_TOKEN` | *(dari akun gofile.io)* | Upload file ke Gofile |
| `SUBSOURCE_API_KEY` | *(subsource.net → Profile → API Key)* | Downloader subtitle `haru-sub` |
| `GDRIVE_CLIENT_ID` | *(Google Cloud)* | Upload GDrive via API |
| `GDRIVE_CLIENT_SECRET` | *(Google Cloud)* | Upload GDrive via API |
| `GDRIVE_REFRESH_TOKEN` | *(OAuth flow)* | Upload GDrive via API |
| `GDRIVE_FOLDER_ID` | *(opsional)* | Folder GDrive tujuan |

## 1 — Install AIO Downloader


In [ ]:
#@title Install AIO { display-mode: "form" }
install_aio = True #@param {type:"boolean"}

if install_aio:
    import subprocess, os, base64, json
    print('Install system (ffmpeg, yt-dlp, requests, colorama, gdown)...')
    subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'tmux', 'curl', 'tree', 'wget'], capture_output=True)
    subprocess.run(['pip', 'install', '-q', 'yt-dlp', 'requests', 'colorama', 'gdown', 'cloudscraper'], capture_output=True)
    print('Pasang commands...')
    TOOLS = {
        'haru-lrc': """...""",
        'haru-manga': """...""",
        'haru-ytdl': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LXl0ZGwgLSBZb3VUdWJlIGRvd25sb2FkZXIgKHBvcnQgb2YgeW91dHViZV9kb3dubG9hZGVyLmJhdCBtZW51KS4iIiIKaW1wb3J0IHN1YnByb2Nlc3MsIHN5cywgb3MsIHJlLCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKQkFTRSA9IFBhdGgoJy9jb250ZW50L2Rvd25sb2FkcycpClZJRCA9IEJBU0UgLyAnVmlkZW8nCkFVRCA9IEJBU0UgLyAnQXVkaW8nClBMID0gQkFTRSAvICdQbGF5bGlzdCcKZm9yIGQgaW4gW1ZJRCwgQVVELCBQTF06CiAgICBkLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKQ09PS0lFUyA9IFBhdGgoJy9jb250ZW50L2Nvb2tpZXMudHh0JykKClRHQk9UID0gJycKT1dORVIgPSAnJwoKZGVmIGxvYWRfc2VjcmV0cygpOgogICAgZ2xvYmFsIFRHQk9ULCBPV05FUgogICAgdHJ5OgogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKToKICAgICAgICAgICAgZCA9IGpzb24ubG9hZChvcGVuKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKSkKICAgICAgICAgICAgZm9yIGssIHYgaW4gZC5pdGVtcygpOgogICAgICAgICAgICAgICAgaWYgdiBhbmQgbm90IG9zLmVudmlyb24uZ2V0KGspOgogICAgICAgICAgICAgICAgICAgIG9zLmVudmlyb25ba10gPSBzdHIodikKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwogICAgVEdCT1QgPSBvcy5lbnZpcm9uLmdldCgnSEFSVV9CT1RfVE9LRU4nLCAnJykKICAgIE9XTkVSID0gb3MuZW52aXJvbi5nZXQoJ09XTkVSX0lEJywgJycpCiAgICBpZiBub3QgVEdCT1Qgb3Igbm90IE9XTkVSOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgICAgIGlmIG5vdCBUR0JPVDoKICAgICAgICAgICAgICAgIFRHQk9UID0gc3RyKHVzZXJkYXRhLmdldCgnSEFSVV9CT1RfVE9LRU4nKSBvciAnJykKICAgICAgICAgICAgaWYgbm90IE9XTkVSOgogICAgICAgICAgICAgICAgT1dORVIgPSBzdHIodXNlcmRhdGEuZ2V0KCdPV05FUl9JRCcpIG9yICcnKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKCmRlZiB0Z19zZW5kKG1zZyk6CiAgICBpZiBub3QgVEdCT1Qgb3Igbm90IE9XTkVSOgogICAgICAgIHJldHVybgogICAgdHJ5OgogICAgICAgIGltcG9ydCByZXF1ZXN0cwogICAgICAgIHJlcXVlc3RzLnBvc3QoJ2h0dHBzOi8vYXBpLnRlbGVncmFtLm9yZy9ib3QnICsgVEdCT1QgKyAnL3NlbmRNZXNzYWdlJywKICAgICAgICAgICAgICAgICAgICAgIGpzb249eydjaGF0X2lkJzogT1dORVIsICd0ZXh0JzogbXNnLCAncGFyc2VfbW9kZSc6ICdIVE1MJywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICdkaXNhYmxlX3dlYl9wYWdlX3ByZXZpZXcnOiBUcnVlfSwgdGltZW91dD0xMCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcGFzcwoKZGVmIGNpKCk6CiAgICBvcy5zeXN0ZW0oJ2NscycgaWYgb3MubmFtZSA9PSAnbnQnIGVsc2UgJ2NsZWFyJykKCmRlZiBoZHIodCk6CiAgICBwcmludCgnXG4nICsgJz0nICogNjIpCiAgICBwcmludCgnICAnICsgdCkKICAgIHByaW50KCc9JyAqIDYyKQoKZGVmIGNvb2tpZV9hcmdzKCk6CiAgICBpZiBDT09LSUVTLmV4aXN0cygpOgogICAgICAgIHVzZSA9IGlucHV0KCcgIGNvb2tpZXMudHh0IGRpdGVtdWthbiwgcGFrYWk/IFtZL25dOiAnKS5zdHJpcCgpLmxvd2VyKCkKICAgICAgICBpZiB1c2UgaW4gKCcnLCAneScpOgogICAgICAgICAgICByZXR1cm4gWyctLWNvb2tpZXMnLCBzdHIoQ09PS0lFUyldCiAgICByZXR1cm4gW10KCmRlZiBwcmV2aWV3KHVybCk6CiAgICBwcmludCgnICBDZWsgaW5mby4uLicpCiAgICByID0gc3VicHJvY2Vzcy5ydW4oWyd5dC1kbHAnLCAnLS1uby1wbGF5bGlzdCcsICctLXByaW50JywKICAgICAgICAgICAgICAgICAgICAgICAgJ0p1ZHVsOiAlKHRpdGxlKXMgfCBEdXJhc2k6ICUoZHVyYXRpb25fc3RyaW5nKXMgfCBDaGFubmVsOiAlKHVwbG9hZGVyKXMnLAogICAgICAgICAgICAgICAgICAgICAgICAnLS1za2lwLWRvd25sb2FkJywgdXJsXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTYwKQogICAgcHJpbnQoJyAgJyArIChyLnN0ZG91dC5zdHJpcCgpIG9yICd0aWRhayBiaXNhIHByZXZpZXcnKSArICdcbicpCgpkZWYgYXNrX3Jlc29sdXRpb24oKToKICAgIHByaW50KCcgIFJlc29sdXNpOiBbMV0gQmVzdCAgWzJdIDEwODBwICBbM10gNzIwcCAgWzRdIDQ4MHAnKQogICAgYyA9IChpbnB1dCgnICBQaWxpaCBbMS00XSAoRW50ZXI9MSk6ICcpLnN0cmlwKCkgb3IgJzEnKQogICAgaWYgYyA9PSAnMic6CiAgICAgICAgcmV0dXJuICdidipbaGVpZ2h0PD0xMDgwXStiYS9iW2hlaWdodDw9MTA4MF0vYnYqK2JhL2InCiAgICBpZiBjID09ICczJzoKICAgICAgICByZXR1cm4gJ2J2KltoZWlnaHQ8PTcyMF0rYmEvYltoZWlnaHQ8PTcyMF0vYnYqK2JhL2InCiAgICBpZiBjID09ICc0JzoKICAgICAgICByZXR1cm4gJ2J2KltoZWlnaHQ8PTQ4MF0rYmEvYltoZWlnaHQ8PTQ4MF0vYnYqK2JhL2InCiAgICByZXR1cm4gJ2J2KitiYS9iJwoKZGVmIGFza19tZXJnZSgpOgogICAgcHJpbnQoJyAgRm9ybWF0OiBbMV0gbXA0ICBbMl0gbWt2JykKICAgIGMgPSAoaW5wdXQoJyAgUGlsaWggWzEtMl0gKEVudGVyPTEpOiAnKS5zdHJpcCgpIG9yICcxJykKICAgIHJldHVybiAnbWt2JyBpZiBjID09ICcyJyBlbHNlICdtcDQnCgpkZWYgYXNrX3N1YnRpdGxlKCk6CiAgICBwcmludCgnICBTdWJ0aXRsZTogWzFdIFRhbnBhICBbMl0gRW1iZWQgIFszXSBGaWxlIHBpc2FoICBbNF0gS2VkdWFueWEnKQogICAgYyA9IChpbnB1dCgnICBQaWxpaCBbMS00XSAoRW50ZXI9MSk6ICcpLnN0cmlwKCkgb3IgJzEnKQogICAgaWYgYyA9PSAnMSc6CiAgICAgICAgcmV0dXJuIFtdCiAgICBsYW5ncyA9IGlucHV0KCcgIEJhaGFzYSAoaWQsZW4samEgLyBhbGwpIFthbGxdOiAnKS5zdHJpcCgpIG9yICdhbGwnCiAgICBhcmdzID0gWyctLXN1Yi1sYW5ncycsIGxhbmdzXQogICAgaWYgYyBpbiAoJzInLCAnNCcpOgogICAgICAgIGFyZ3MuYXBwZW5kKCctLWVtYmVkLXN1YnMnKQogICAgaWYgYyBpbiAoJzMnLCAnNCcpOgogICAgICAgIGFyZ3MuYXBwZW5kKCctLXdyaXRlLXN1YnMnKQogICAgcmV0dXJuIGFyZ3MKCmRlZiBydW5fZGwoYXJncywgZGVzYyk6CiAgICBwcmludCgnXG4gID09PSBET1dOTE9BRDogJyArIGRlc2MgKyAnID09PVxuJykKICAgIHIgPSBzdWJwcm9jZXNzLnJ1bihbJ3l0LWRscCddICsgYXJncykKICAgIGlmIHIucmV0dXJuY29kZSA9PSAwOgogICAgICAgIHByaW50KCdcbiAgU2VsZXNhaS4nKQogICAgICAgIHRnX3NlbmQoJzxiPnl0ZGwgc2VsZXNhaTwvYj5cbicgKyBkZXNjKQogICAgZWxzZToKICAgICAgICBwcmludCgnXG4gIEdhZ2FsIC8gZGliYXRhbGthbi4nKQogICAgaW5wdXQoJ1xuICBFbnRlci4uLicpCgpkZWYgc2luZ2xlX3ZpZGVvKCk6CiAgICBjaSgpCiAgICBoZHIoJ1ZJREVPIFNBVFVBTicpCiAgICB1cmwgPSBpbnB1dCgnXG4gIFVSTDogJykuc3RyaXAoKQogICAgaWYgbm90IHVybDoKICAgICAgICByZXR1cm4KICAgIHByZXZpZXcodXJsKQogICAgZm10ID0gYXNrX3Jlc29sdXRpb24oKQogICAgbWVyZ2UgPSBhc2tfbWVyZ2UoKQogICAgc3VicyA9IGFza19zdWJ0aXRsZSgpCiAgICBjayA9IGNvb2tpZV9hcmdzKCkKICAgIG91dCA9IHN0cihWSUQgLyAnJSh0aXRsZSlzIFslKGlkKXNdLiUoZXh0KXMnKQogICAgcnVuX2RsKFsnLS1uby1wbGF5bGlzdCddICsgY2sgKyBbJy1mJywgZm10LCAnLS1tZXJnZS1vdXRwdXQtZm9ybWF0JywgbWVyZ2VdICsKICAgICAgICAgICBzdWJzICsgWyctbycsIG91dCwgdXJsXSwgJ3ZpZGVvOiAnICsgdXJsKQoKZGVmIGF1ZGlvX29ubHkocGxheWxpc3Q9RmFsc2UpOgogICAgY2koKQogICAgaGRyKCdBVURJTyBTQUpBJyArICgnIChQTEFZTElTVCknIGlmIHBsYXlsaXN0IGVsc2UgJycpKQogICAgdXJsID0gaW5wdXQoJ1xuICBVUkw6ICcpLnN0cmlwKCkKICAgIGlmIG5vdCB1cmw6CiAgICAgICAgcmV0dXJuCiAgICBwcmludCgnICBGb3JtYXQ6IFsxXSBtcDMgIFsyXSBtNGEgIFszXSBvcHVzICBbNF0gZmxhYyAgWzVdIHdhdicpCiAgICBjID0gKGlucHV0KCcgIFBpbGloIChFbnRlcj0xKTogJykuc3RyaXAoKSBvciAnMScpCiAgICBleHQgPSB7JzEnOiAnbXAzJywgJzInOiAnbTRhJywgJzMnOiAnb3B1cycsICc0JzogJ2ZsYWMnLCAnNSc6ICd3YXYnfS5nZXQoYywgJ21wMycpCiAgICBwcmludCgnICBLdWFsaXRhczogWzFdIEJlc3QgIFsyXSAzMjBLICBbM10gMjU2SyAgWzRdIDE5MksgIFs1XSAxMjhLJykKICAgIHEgPSAoaW5wdXQoJyAgUGlsaWggKEVudGVyPTEpOiAnKS5zdHJpcCgpIG9yICcxJykKICAgIHF1YWwgPSB7JzEnOiAnMCcsICcyJzogJzMyMEsnLCAnMyc6ICcyNTZLJywgJzQnOiAnMTkySycsICc1JzogJzEyOEsnfS5nZXQocSwgJzAnKQogICAgY2sgPSBjb29raWVfYXJncygpCiAgICBvdXRkaXIgPSBQTCBpZiBwbGF5bGlzdCBlbHNlIEFVRAogICAgb3V0ID0gc3RyKG91dGRpciAvICclKHRpdGxlKXMgWyUoaWQpc10uJShleHQpcycpCiAgICBhcmdzID0gKFtdIGlmIHBsYXlsaXN0IGVsc2UgWyctLW5vLXBsYXlsaXN0J10pICsgY2sgKyBbJy0tZXh0cmFjdC1hdWRpbycsCiAgICAgICAgICAgICctLWF1ZGlvLWZvcm1hdCcsIGV4dCwgJy0tYXVkaW8tcXVhbGl0eScsIHF1YWwsCiAgICAgICAgICAgICctLWVtYmVkLXRodW1ibmFpbCcsICctLWFkZC1tZXRhZGF0YScsICctbycsIG91dCwgdXJsXQogICAgcnVuX2RsKGFyZ3MsICdhdWRpbyAnICsgZXh0ICsgJzogJyArIHVybCkKCmRlZiBwbGF5bGlzdF92aWRlbygpOgogICAgY2koKQogICAgaGRyKCdQTEFZTElTVCBWSURFTycpCiAgICB1cmwgPSBpbnB1dCgnXG4gIFVSTCBwbGF5bGlzdDogJykuc3RyaXAoKQogICAgaWYgbm90IHVybDoKICAgICAgICByZXR1cm4KICAgIHByaW50KCcgIEFtYmlsIGRhZnRhciBpc2kuLi4nKQogICAgciA9IHN1YnByb2Nlc3MucnVuKFsneXQtZGxwJywgJy0tZmxhdC1wbGF5bGlzdCcsICctLXByaW50JywKICAgICAgICAgICAgICAgICAgICAgICAgJyUocGxheWxpc3RfaW5kZXgpMDNkIHwgJSh0aXRsZSlzIHwgJShkdXJhdGlvbl9zdHJpbmcpcycsIHVybF0sCiAgICAgICAgICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTEyMCkKICAgIHByaW50KHIuc3Rkb3V0WzozMDAwXSkKICAgIHNlbCA9IChpbnB1dCgnICBQaWxpaCBbQSBzZW11YSAvIDEtNSAvIDEsMyw1LTddIChFbnRlcj1BKTogJykuc3RyaXAoKSBvciAnQScpCiAgICBwYXJncyA9IFtdCiAgICBpZiBzZWwudXBwZXIoKSAhPSAnQSc6CiAgICAgICAgcGFyZ3MgPSBbJy0tcGxheWxpc3QtaXRlbXMnLCBzZWxdCiAgICBmbXQgPSBhc2tfcmVzb2x1dGlvbigpCiAgICBtZXJnZSA9IGFza19tZXJnZSgpCiAgICBjayA9IGNvb2tpZV9hcmdzKCkKICAgIG91dCA9IHN0cihQTCAvICclKHBsYXlsaXN0X2luZGV4KTAzZCAtICUodGl0bGUpcyBbJShpZClzXS4lKGV4dClzJykKICAgIHJ1bl9kbChwYXJncyArIGNrICsgWyctZicsIGZtdCwgJy0tbWVyZ2Utb3V0cHV0LWZvcm1hdCcsIG1lcmdlLCAnLW8nLCBvdXQsIHVybF0sCiAgICAgICAgICAgJ3BsYXlsaXN0OiAnICsgdXJsICsgJyBpdGVtcz0nICsgc2VsKQoKZGVmIHN1YnRpdGxlX21vZGUoKToKICAgIGNpKCkKICAgIGhkcignVklERU8gKyBTVUJUSVRMRSBTUEVTSUFMJykKICAgIHVybCA9IGlucHV0KCdcbiAgVVJMOiAnKS5zdHJpcCgpCiAgICBpZiBub3QgdXJsOgogICAgICAgIHJldHVybgogICAgcHJldmlldyh1cmwpCiAgICBsYW5ncyA9IGlucHV0KCcgIEJhaGFzYSAoaWQsZW4samEgLyBhbGwpIFthbGxdOiAnKS5zdHJpcCgpIG9yICdhbGwnCiAgICBmbXQgPSBhc2tfcmVzb2x1dGlvbigpCiAgICBtZXJnZSA9IGFza19tZXJnZSgpCiAgICBjayA9IGNvb2tpZV9hcmdzKCkKICAgIG91dCA9IHN0cihWSUQgLyAnJSh0aXRsZSlzIFslKGlkKXNdLiUoZXh0KXMnKQogICAgcnVuX2RsKFsnLS1uby1wbGF5bGlzdCddICsgY2sgKyBbJy1mJywgZm10LCAnLS1tZXJnZS1vdXRwdXQtZm9ybWF0JywgbWVyZ2UsCiAgICAgICAgICAgICctLXN1Yi1sYW5ncycsIGxhbmdzLCAnLS1lbWJlZC1zdWJzJywgJy0td3JpdGUtc3VicycsICctbycsIG91dCwgdXJsXSwKICAgICAgICAgICAndmlkZW8rc3ViOiAnICsgdXJsKQoKZGVmIGFkdmFuY2VkKCk6CiAgICBjaSgpCiAgICBoZHIoJ01PREUgQURWQU5DRUQnKQogICAgcHJpbnQoJyAgQ29udG9oOiAtLW5vLXBsYXlsaXN0IC1mICJidiorYmEvYiIgVVJMJykKICAgIHJhdyA9IGlucHV0KCdcbiAgQXJndW1lbiB5dC1kbHA6ICcpLnN0cmlwKCkKICAgIGlmIG5vdCByYXc6CiAgICAgICAgcmV0dXJuCiAgICBpbXBvcnQgc2hsZXgKICAgIHJ1bl9kbChzaGxleC5zcGxpdChyYXcpLCAnY3VzdG9tOiAnICsgcmF3Wzo4MF0pCgpkZWYgdXBkYXRlX3l0ZGwoKToKICAgIHByaW50KCcgIFVwZGF0ZSB5dC1kbHAuLi4nKQogICAgc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncGlwJywgJ2luc3RhbGwnLCAnLXEnLCAnLVUnLCAneXQtZGxwJ10pCiAgICByID0gc3VicHJvY2Vzcy5ydW4oWyd5dC1kbHAnLCAnLS12ZXJzaW9uJ10sIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSkKICAgIHByaW50KCcgIFZlcnNpOiAnICsgci5zdGRvdXQuc3RyaXAoKSkKICAgIGlucHV0KCdcbiAgRW50ZXIuLi4nKQoKZGVmIG1haW4oKToKICAgIGxvYWRfc2VjcmV0cygpCiAgICB3aGlsZSBUcnVlOgogICAgICAgIGNpKCkKICAgICAgICBwcmludCgnXG4nICsgJz0nICogNjIpCiAgICAgICAgcHJpbnQoJyAgaGFydS15dGRsIC0tIFlvdVR1YmUgRG93bmxvYWRlcicpCiAgICAgICAgcHJpbnQoJz0nICogNjIpCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCcgIFsxXSBWaWRlbyBzYXR1YW4nKQogICAgICAgIHByaW50KCcgIFsyXSBBdWRpbyBzYWphIChtcDMvbTRhL29wdXMvZmxhYy93YXYpJykKICAgICAgICBwcmludCgnICBbM10gUGxheWxpc3QgdmlkZW8nKQogICAgICAgIHByaW50KCcgIFs0XSBQbGF5bGlzdCBhdWRpbycpCiAgICAgICAgcHJpbnQoJyAgWzVdIFZpZGVvICsgc3VidGl0bGUgc3Blc2lhbCcpCiAgICAgICAgcHJpbnQoJyAgWzZdIEFkdmFuY2VkIChhcmd1bWVuIHNlbmRpcmkpJykKICAgICAgICBwcmludCgnICBbN10gVXBkYXRlIHl0LWRscCcpCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCcgIFtRXSBLZWx1YXInKQogICAgICAgIHByaW50KCkKICAgICAgICBjID0gaW5wdXQoJyAgUGlsaWg6ICcpLnN0cmlwKCkudXBwZXIoKQogICAgICAgIGlmIGMgPT0gJ1EnOgogICAgICAgICAgICBwcmludCgnXG4gIEJ5ZSEnKQogICAgICAgICAgICByZXR1cm4KICAgICAgICBlbGlmIGMgPT0gJzEnOgogICAgICAgICAgICBzaW5nbGVfdmlkZW8oKQogICAgICAgIGVsaWYgYyA9PSAnMic6CiAgICAgICAgICAgIGF1ZGlvX29ubHkoKQogICAgICAgIGVsaWYgYyA9PSAnMyc6CiAgICAgICAgICAgIHBsYXlsaXN0X3ZpZGVvKCkKICAgICAgICBlbGlmIGMgPT0gJzQnOgogICAgICAgICAgICBhdWRpb19vbmx5KHBsYXlsaXN0PVRydWUpCiAgICAgICAgZWxpZiBjID09ICc1JzoKICAgICAgICAgICAgc3VidGl0bGVfbW9kZSgpCiAgICAgICAgZWxpZiBjID09ICc2JzoKICAgICAgICAgICAgYWR2YW5jZWQoKQogICAgICAgIGVsaWYgYyA9PSAnNyc6CiAgICAgICAgICAgIHVwZGF0ZV95dGRsKCkKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICB0cnk6CiAgICAgICAgbWFpbigpCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgcHJpbnQoJ1xuICBEaWJhdGFsa2FuLicpCg==""",
        'haru-check': """...""",
        'haru-transferit': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LXRyYW5zZmVyaXQgLSBUcmFuc2Zlci5pdCBBdXRvLVJlbmV3IENMSSAocG9ydCBUcmFuc2Zlcml0X1JlbmV3YWwpLgpQZXJwYW5qYW5nICYgaGlkdXBrYW4ga2VtYmFsaSBTRU1VQSB0cmFuc2ZlciBkaSB0cmFuc2Zlci5pdCAoYmFja2VuZCBNRUdBKS4KVGFucGEgZGVwZW5kZW5jeSByaWNoLCBjdWt1cCByZXF1ZXN0cy4KIiIiCmltcG9ydCBiYXNlNjQKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByYW5kb20KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcmVxdWVzdHMKCkNPTkZJR19QQVRIID0gUGF0aCgiL2NvbnRlbnQvLmhhcnVfdHJhbnNmZXJpdC5qc29uIikKCkRFRkFVTFRTID0gewogICAgInNpZCI6ICIiLAogICAgImRheXMiOiA5MCwKICAgICJkZWxheV9taW4iOiAwLjYsCiAgICAiZGVsYXlfbWF4IjogMS40LAogICAgImFwaV91cmwiOiAiaHR0cHM6Ly9idDcuYXBpLm1lZ2EuY28ubnovY3MiLAogICAgInVzZXJfYWdlbnQiOiAoCiAgICAgICAgIk1vemlsbGEvNS4wIChXaW5kb3dzIE5UIDEwLjA7IFdpbjY0OyB4NjQpICIKICAgICAgICAiQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgIgogICAgICAgICJDaHJvbWUvMTI3LjAuMC4wIFNhZmFyaS81MzcuMzYiCiAgICApLAp9CgoKZGVmIGxvYWRfY29uZmlnKCkgLT4gZGljdDoKICAgIGNvbmZpZyA9IGRpY3QoREVGQVVMVFMpCiAgICBpZiBDT05GSUdfUEFUSC5leGlzdHMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGNvbmZpZy51cGRhdGUoanNvbi5sb2FkcyhDT05GSUdfUEFUSC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgIHJldHVybiBjb25maWcKCgpkZWYgc2F2ZV9jb25maWcoY29uZmlnOiBkaWN0KSAtPiBOb25lOgogICAgQ09ORklHX1BBVEgud3JpdGVfdGV4dChqc29uLmR1bXBzKGNvbmZpZywgaW5kZW50PTQsIGVuc3VyZV9hc2NpaT1GYWxzZSksIGVuY29kaW5nPSJ1dGYtOCIpCgoKY2xhc3MgTWVnYUFQSUVycm9yKFJ1bnRpbWVFcnJvcik6CiAgICBwYXNzCgoKZGVmIGI2NHVybF9kZWNvZGUoZGF0YTogc3RyKSAtPiBieXRlczoKICAgIHBhZGRpbmcgPSAiPSIgKiAoLWxlbihkYXRhKSAlIDQpCiAgICByZXR1cm4gYmFzZTY0LnVybHNhZmVfYjY0ZGVjb2RlKGRhdGEgKyBwYWRkaW5nKQoKCmNsYXNzIFRyYW5zZmVyaXRNYW5hZ2VyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNpZDogc3RyLCBjb25maWc6IGRpY3QpOgogICAgICAgIGlmIG5vdCBzaWQgb3Igbm90IHNpZC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTSUQga29zb25nLiBNYXN1a2thbiBTZXNzaW9uIElEIChsb2NhbFN0b3JhZ2UgJ3NpZCcpLiIpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcgb3IgbG9hZF9jb25maWcoKQogICAgICAgIHNlbGYuc2lkID0gc2lkLnN0cmlwKCkKICAgICAgICBzZWxmLnNlcW5vID0gcmFuZG9tLnJhbmRpbnQoMTAwXzAwMCwgOTk5Xzk5OSkKICAgICAgICBzZWxmLnNlc3Npb24gPSByZXF1ZXN0cy5TZXNzaW9uKCkKICAgICAgICBzZWxmLnNlc3Npb24uaGVhZGVycy51cGRhdGUoewogICAgICAgICAgICAiVXNlci1BZ2VudCI6IHNlbGYuY29uZmlnLmdldCgidXNlcl9hZ2VudCIpLAogICAgICAgICAgICAiQWNjZXB0IjogImFwcGxpY2F0aW9uL2pzb24sIHRleHQvcGxhaW4sICovKiIsCiAgICAgICAgICAgICJPcmlnaW4iOiAiaHR0cHM6Ly90cmFuc2Zlci5pdCIsCiAgICAgICAgICAgICJSZWZlcmVyIjogImh0dHBzOi8vdHJhbnNmZXIuaXQvIiwKICAgICAgICAgICAgIkNvbnRlbnQtVHlwZSI6ICJhcHBsaWNhdGlvbi9qc29uIiwKICAgICAgICB9KQoKICAgIGRlZiBfYXBpX3VybChzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5jb25maWcuZ2V0KCJhcGlfdXJsIiwgImh0dHBzOi8vYnQ3LmFwaS5tZWdhLmNvLm56L2NzIikKCiAgICBkZWYgX3JlcShzZWxmLCBwYXlsb2FkKToKICAgICAgICBzZWxmLnNlcW5vICs9IDEKICAgICAgICBwYXJhbXMgPSB7ImlkIjogc2VsZi5zZXFubywgInNpZCI6IHNlbGYuc2lkfQogICAgICAgIGJvZHkgPSBwYXlsb2FkIGlmIGlzaW5zdGFuY2UocGF5bG9hZCwgbGlzdCkgZWxzZSBbcGF5bG9hZF0KICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSg0KToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcmVzcCA9IHNlbGYuc2Vzc2lvbi5wb3N0KHNlbGYuX2FwaV91cmwoKSwgcGFyYW1zPXBhcmFtcywganNvbj1ib2R5LCB0aW1lb3V0PTMwKQogICAgICAgICAgICAgICAgcmVzcC5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICAgICAgICAgIGRhdGEgPSByZXNwLmpzb24oKQogICAgICAgICAgICBleGNlcHQgKHJlcXVlc3RzLlJlcXVlc3RFeGNlcHRpb24sIFZhbHVlRXJyb3IpIGFzIGV4YzoKICAgICAgICAgICAgICAgIGlmIGF0dGVtcHQgPCAzOgogICAgICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAoMSArIGF0dGVtcHQpCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJhaXNlIE1lZ2FBUElFcnJvcihmIk5ldHdvcmsvSFRUUCBlcnJvcjoge2V4Y30iKSBmcm9tIGV4YwogICAgICAgICAgICBjb2RlID0gZGF0YSBpZiBpc2luc3RhbmNlKGRhdGEsIGludCkgZWxzZSAoCiAgICAgICAgICAgICAgICBkYXRhWzBdIGlmIGlzaW5zdGFuY2UoZGF0YSwgbGlzdCkgYW5kIGxlbihkYXRhKSA9PSAxIGFuZCBpc2luc3RhbmNlKGRhdGFbMF0sIGludCkgZWxzZSBOb25lCiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgY29kZSBpcyBub3QgTm9uZSBhbmQgY29kZSA8IDA6CiAgICAgICAgICAgICAgICBpZiBjb2RlID09IC0zIGFuZCBhdHRlbXB0IDwgMzoKICAgICAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDEgKyBhdHRlbXB0KQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICByYWlzZSBNZWdhQVBJRXJyb3IoZiJNRUdBIEFQSSBFcnJvciBDb2RlOiB7Y29kZX0iKQogICAgICAgICAgICByZXR1cm4gZGF0YVswXSBpZiBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpIGVsc2UgZGF0YQogICAgICAgIHJhaXNlIE1lZ2FBUElFcnJvcigiUmVxdWVzdCBnYWdhbCBzZXRlbGFoIGJlYmVyYXBhIHBlcmNvYmFhbi4iKQoKICAgIGRlZiBsaXN0X3RyYW5zZmVycyhzZWxmKToKICAgICAgICBkYXRhID0gc2VsZi5fcmVxKHsiYSI6ICJ4bCJ9KQogICAgICAgIGlmIGlzaW5zdGFuY2UoZGF0YSwgbGlzdCk6CiAgICAgICAgICAgIHJldHVybiBbdCBmb3IgdCBpbiBkYXRhIGlmIGlzaW5zdGFuY2UodCwgZGljdCldCiAgICAgICAgcmV0dXJuIFtdCgogICAgZGVmIHJlbmV3X2FuZF9yZXZpdmUoc2VsZiwgeGgsIGRheXM9OTApOgogICAgICAgIHJldHVybiBzZWxmLl9yZXEoeyJhIjogInhtIiwgInhoIjogeGgsICJlIjogZGF5cyAqIDg2NDAwfSkKCiAgICBkZWYgZ2V0X21ldGFkYXRhKHNlbGYsIHhoKToKICAgICAgICBkYXRhID0gc2VsZi5fcmVxKHsiYSI6ICJ4aSIsICJ4aCI6IHhofSkKICAgICAgICByZXR1cm4gZGF0YSBpZiBpc2luc3RhbmNlKGRhdGEsIGRpY3QpIGVsc2Uge30KCiAgICBkZWYgZGVsZXRlX3RyYW5zZmVyKHNlbGYsIHhoKToKICAgICAgICByZXR1cm4gc2VsZi5fcmVxKHsiYSI6ICJ4ZCIsICJ4aCI6IHhofSkKCiAgICBkZWYgcmVuZXdfYWxsKHNlbGYsIGRheXM9OTAsIGRlbGF5X3JhbmdlPU5vbmUsIG9uX3Byb2dyZXNzPU5vbmUpOgogICAgICAgIGlmIGRlbGF5X3JhbmdlIGlzIE5vbmU6CiAgICAgICAgICAgIGRlbGF5X3JhbmdlID0gKAogICAgICAgICAgICAgICAgZmxvYXQoc2VsZi5jb25maWcuZ2V0KCJkZWxheV9taW4iLCAwLjYpKSwKICAgICAgICAgICAgICAgIGZsb2F0KHNlbGYuY29uZmlnLmdldCgiZGVsYXlfbWF4IiwgMS40KSksCiAgICAgICAgICAgICkKICAgICAgICB0cmFuc2ZlcnMgPSBzZWxmLmxpc3RfdHJhbnNmZXJzKCkKICAgICAgICByZXN1bHRzID0geyJ0b3RhbCI6IGxlbih0cmFuc2ZlcnMpLCAic3VjY2VzcyI6IDAsICJmYWlsZWQiOiAwLCAiZGV0YWlscyI6IFtdfQogICAgICAgIGZvciBpZHgsIHQgaW4gZW51bWVyYXRlKHRyYW5zZmVycywgMSk6CiAgICAgICAgICAgIHhoID0gdC5nZXQoInhoIiwgIiIpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHNlbGYucmVuZXdfYW5kX3Jldml2ZSh4aCwgZGF5cz1kYXlzKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIG1ldGEgPSBzZWxmLmdldF9tZXRhZGF0YSh4aCkKICAgICAgICAgICAgICAgICAgICByYXdfdGl0bGUgPSBtZXRhLmdldCgidCIsICIiKQogICAgICAgICAgICAgICAgICAgIHRpdGxlID0gYjY0dXJsX2RlY29kZShyYXdfdGl0bGUpLmRlY29kZSgidXRmLTgiLCBlcnJvcnM9Imlnbm9yZSIpIGlmIHJhd190aXRsZSBlbHNlICJObyBUaXRsZSIKICAgICAgICAgICAgICAgICAgICB0b3RhbF9ieXRlcyA9IG1ldGEuZ2V0KCJzaXplIiwgWzBdKVswXSBpZiBpc2luc3RhbmNlKG1ldGEuZ2V0KCJzaXplIiksIGxpc3QpIGVsc2UgMAogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICB0aXRsZSA9ICJBY3RpdmUgKE1ldGFkYXRhIHVuYXZhaWxhYmxlKSIKICAgICAgICAgICAgICAgICAgICB0b3RhbF9ieXRlcyA9IDAKICAgICAgICAgICAgICAgIHJlc3VsdHNbInN1Y2Nlc3MiXSArPSAxCiAgICAgICAgICAgICAgICByZXN1bHRzWyJkZXRhaWxzIl0uYXBwZW5kKCh4aCwgdGl0bGUsIHRvdGFsX2J5dGVzLCBmIkFDVElWRSAoe2RheXN9IERheXMpIikpCiAgICAgICAgICAgICAgICBpZiBvbl9wcm9ncmVzczoKICAgICAgICAgICAgICAgICAgICBvbl9wcm9ncmVzcyhpZHgsIGxlbih0cmFuc2ZlcnMpLCB4aCwgdGl0bGUsICJTVUNDRVNTIikKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICByZXN1bHRzWyJmYWlsZWQiXSArPSAxCiAgICAgICAgICAgICAgICByZXN1bHRzWyJkZXRhaWxzIl0uYXBwZW5kKCh4aCwgIkVycm9yIiwgMCwgc3RyKGV4YykpKQogICAgICAgICAgICAgICAgaWYgb25fcHJvZ3Jlc3M6CiAgICAgICAgICAgICAgICAgICAgb25fcHJvZ3Jlc3MoaWR4LCBsZW4odHJhbnNmZXJzKSwgeGgsICJFcnJvciIsIHN0cihleGMpKQogICAgICAgICAgICBpZiBpZHggPCBsZW4odHJhbnNmZXJzKToKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAocmFuZG9tLnVuaWZvcm0oKmRlbGF5X3JhbmdlKSkKICAgICAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBvayh0KTogcmV0dXJuICdcMDMzWzkybScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBlcih0KTogcmV0dXJuICdcMDMzWzkxbScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBkaW0odCk6IHJldHVybiAnXDAzM1s5MG0nICsgdCArICdcMDMzWzBtJwpkZWYgY3kodCk6IHJldHVybiAnXDAzM1s5Nm0nICsgdCArICdcMDMzWzBtJwpkZWYgeWwodCk6IHJldHVybiAnXDAzM1s5M20nICsgdCArICdcMDMzWzBtJwoKZGVmIGNpKCk6CiAgICBpbXBvcnQgc3lzCiAgICBzeXMuc3Rkb3V0LndyaXRlKCdceDFiWzJKXHgxYltIJykKICAgIHN5cy5zdGRvdXQuZmx1c2goKQoKZGVmIGhkcih0aXRsZSk6CiAgICBwcmludCgnXG4nICsgJz0nICogNjApCiAgICBwcmludCgnICAnICsgdGl0bGUpCiAgICBwcmludCgnPScgKiA2MCkKCgpkZWYgaHVtYW5fc2l6ZShudW0pOgogICAgZm9yIHVuaXQgaW4gKCJCIiwgIktCIiwgIk1CIiwgIkdCIiwgIlRCIik6CiAgICAgICAgaWYgYWJzKG51bSkgPCAxMDI0LjA6CiAgICAgICAgICAgIHJldHVybiBmIntudW0gaWYgdW5pdCA9PSAnQicgZWxzZSBmJ3tudW06LjFmfSd9IHt1bml0fSIKICAgICAgICBudW0gLz0gMTAyNC4wCiAgICByZXR1cm4gZiJ7bnVtOi4xZn0gUEIiCgoKZGVmIGVuc3VyZV9zaWQoY29uZmlnKToKICAgIHNpZCA9IGNvbmZpZy5nZXQoInNpZCIsICIiKS5zdHJpcCgpCiAgICBpZiBzaWQ6CiAgICAgICAgcmV0dXJuIHNpZAogICAgcHJpbnQoeWwoIlxuICBTZXNzaW9uIElEIChTSUQpIGJlbHVtIHRlcnNpbXBhbi4iKSkKICAgIHByaW50KGRpbSgiICBDYXJhIGRhcGF0OiBidWthIHRyYW5zZmVyLml0IGRpIGJyb3dzZXIsIERldlRvb2xzIChGMTIpIC0+IEFwcGxpY2F0aW9uXG4gIC0+IExvY2FsIFN0b3JhZ2UgLT4gc2FsaW4gbmlsYWkgJ3NpZCcuIikpCiAgICBzaWQgPSBpbnB1dChjeSgiICBNYXN1a2thbiBTZXNzaW9uIElEOiAiKSkuc3RyaXAoKQogICAgd2hpbGUgbm90IHNpZDoKICAgICAgICBwcmludChlcigiICBTSUQgdGlkYWsgYm9sZWgga29zb25nLiIpKQogICAgICAgIHNpZCA9IGlucHV0KGN5KCIgIE1hc3Vra2FuIFNlc3Npb24gSUQ6ICIpKS5zdHJpcCgpCiAgICBjb25maWdbInNpZCJdID0gc2lkCiAgICBzYXZlX2NvbmZpZyhjb25maWcpCiAgICBwcmludChvaygiICBTSUQgYmVyaGFzaWwgZGlzaW1wYW4uIikpCiAgICByZXR1cm4gc2lkCgoKZGVmIGNtZF9yZW5ld19hbGwoY29uZmlnKToKICAgIGRheXNfaW4gPSBpbnB1dChmIiAgQmVyYXBhIGhhcmkgbWFzYSBha3RpZj8gW3tjb25maWcuZ2V0KCdkYXlzJywgOTApfV06ICIpLnN0cmlwKCkKICAgIGRheXMgPSBpbnQoZGF5c19pbikgaWYgZGF5c19pbi5sc3RyaXAoJy0nKS5pc2RpZ2l0KCkgZWxzZSBpbnQoY29uZmlnLmdldCgnZGF5cycsIDkwKSkKICAgIHNpZCA9IGVuc3VyZV9zaWQoY29uZmlnKQogICAgbWFuYWdlciA9IFRyYW5zZmVyaXRNYW5hZ2VyKHNpZCwgY29uZmlnKQogICAgcHJpbnQoZGltKCIgIE1lbmdhbWJpbCBkYWZ0YXIgdHJhbnNmZXIuLi4iKSkKICAgIHRyeToKICAgICAgICB0cmFuc2ZlcnMgPSBtYW5hZ2VyLmxpc3RfdHJhbnNmZXJzKCkKICAgIGV4Y2VwdCBNZWdhQVBJRXJyb3IgYXMgZXhjOgogICAgICAgIHByaW50KGVyKGYiICBHYWdhbCBtZW5nYW1iaWwgdHJhbnNmZXI6IHtleGN9IikpCiAgICAgICAgcmV0dXJuCiAgICBpZiBub3QgdHJhbnNmZXJzOgogICAgICAgIHByaW50KHlsKCIgIFRpZGFrIGFkYSB0cmFuc2ZlciB5YW5nIGRpdGVtdWthbiBkaSBha3VuLiIpKQogICAgICAgIHJldHVybgogICAgcHJpbnQoZGltKGYiICBEaXRlbXVrYW4ge2xlbih0cmFuc2ZlcnMpfSB0cmFuc2Zlci4gTXVsYWkgcmVuZXcuLi4iKSkKICAgIGRldGFpbHMgPSBbXQogICAgdG90YWwgPSBsZW4odHJhbnNmZXJzKQogICAgZGVmIG9uX3Byb2dyZXNzKGlkeCwgdCwgeGgsIHRpdGxlLCBzdGF0dXMpOgogICAgICAgIGRldGFpbHMuYXBwZW5kKCh4aCwgdGl0bGUsIDAsIHN0YXR1cykpCiAgICAgICAgcHJpbnQoZiIgIFt7aWR4fS97dH1dIHt0aXRsZVs6NDBdfSAtPiB7c3RhdHVzfSIpCiAgICB0cnk6CiAgICAgICAgcmVzdWx0cyA9IG1hbmFnZXIucmVuZXdfYWxsKGRheXM9ZGF5cywgb25fcHJvZ3Jlc3M9b25fcHJvZ3Jlc3MpCiAgICBleGNlcHQgTWVnYUFQSUVycm9yIGFzIGV4YzoKICAgICAgICBwcmludChlcihmIiAgRXJyb3IgZmF0YWw6IHtleGN9IikpCiAgICAgICAgcmV0dXJuCiAgICBwcmludCgpCiAgICBwcmludChvayhmIiAgQmVyaGFzaWw6IHtyZXN1bHRzWydzdWNjZXNzJ119ICAgIikgKyBlcihmIkdhZ2FsOiB7cmVzdWx0c1snZmFpbGVkJ119ICAgIikgKyBmIlRvdGFsOiB7cmVzdWx0c1sndG90YWwnXX0iKQogICAgaWYgcmVzdWx0c1siZGV0YWlscyJdOgogICAgICAgIHByaW50KCdcbicgKyAnPScgKiA2MCkKICAgICAgICBwcmludCgnICBIQVNJTCBSRU5FVyAnICsgZiIoe2RheXN9IEhhcmkpIikKICAgICAgICBwcmludCgnPScgKiA2MCkKICAgICAgICBmb3IgaSwgKHhoLCB0aXRsZSwgc2l6ZSwgc3RhdHVzKSBpbiBlbnVtZXJhdGUocmVzdWx0c1siZGV0YWlscyJdLCAxKToKICAgICAgICAgICAgc2l6ZV9zdHIgPSBodW1hbl9zaXplKHNpemUpIGlmIHNpemUgZWxzZSAiLSIKICAgICAgICAgICAgYmFkZ2UgPSBvayhzdGF0dXMpIGlmIHN0YXR1cy5zdGFydHN3aXRoKCJBQ1RJVkUiKSBlbHNlIGVyKHN0YXR1cykKICAgICAgICAgICAgcHJpbnQoZiIgIHtpOj4yfS4ge3hofSAge3RpdGxlWzo0MF06PDQwfSB7c2l6ZV9zdHI6Pjh9ICB7YmFkZ2V9IikKICAgIGlucHV0KCdcbiAgRW50ZXIuLi4nKQoKCmRlZiBjbWRfbGlzdF90cmFuc2ZlcnMoY29uZmlnKToKICAgIHNpZCA9IGVuc3VyZV9zaWQoY29uZmlnKQogICAgbWFuYWdlciA9IFRyYW5zZmVyaXRNYW5hZ2VyKHNpZCwgY29uZmlnKQogICAgcHJpbnQoZGltKCIgIE1lbmdhbWJpbCBkYWZ0YXIgdHJhbnNmZXIuLi4iKSkKICAgIHRyeToKICAgICAgICB0cmFuc2ZlcnMgPSBtYW5hZ2VyLmxpc3RfdHJhbnNmZXJzKCkKICAgIGV4Y2VwdCBNZWdhQVBJRXJyb3IgYXMgZXhjOgogICAgICAgIHByaW50KGVyKGYiICBHYWdhbCBtZW5nYW1iaWwgdHJhbnNmZXI6IHtleGN9IikpCiAgICAgICAgcmV0dXJuCiAgICBpZiBub3QgdHJhbnNmZXJzOgogICAgICAgIHByaW50KHlsKCIgIFRpZGFrIGFkYSB0cmFuc2ZlciB5YW5nIGRpdGVtdWthbi4iKSkKICAgICAgICByZXR1cm4KICAgIHByaW50KCdcbicgKyAnPScgKiA2MCkKICAgIHByaW50KCcgIERBRlRBUiBUUkFOU0ZFUicpCiAgICBwcmludCgnPScgKiA2MCkKICAgIGZvciB0IGluIHRyYW5zZmVyczoKICAgICAgICB4aCA9IHQuZ2V0KCJ4aCIsICIiKQogICAgICAgIHRpdGxlLCB0b3RhbF9ieXRlcyA9IHhoLCAwCiAgICAgICAgdHJ5OgogICAgICAgICAgICBtZXRhID0gbWFuYWdlci5nZXRfbWV0YWRhdGEoeGgpCiAgICAgICAgICAgIHJhdyA9IG1ldGEuZ2V0KCJ0IiwgIiIpCiAgICAgICAgICAgIGlmIHJhdzoKICAgICAgICAgICAgICAgIHRpdGxlID0gYjY0dXJsX2RlY29kZShyYXcpLmRlY29kZSgidXRmLTgiLCBlcnJvcnM9Imlnbm9yZSIpCiAgICAgICAgICAgIHNpemUgPSBtZXRhLmdldCgic2l6ZSIpCiAgICAgICAgICAgIHRvdGFsX2J5dGVzID0gc2l6ZVswXSBpZiBpc2luc3RhbmNlKHNpemUsIGxpc3QpIGFuZCBzaXplIGVsc2UgMAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHBhc3MKICAgICAgICB0cyA9IHQuZ2V0KCJ0cyIsICIiKQogICAgICAgIHByaW50KGYiICB7eGh9ICB7dGl0bGVbOjQwXTo8NDB9IHtodW1hbl9zaXplKHRvdGFsX2J5dGVzKSBpZiB0b3RhbF9ieXRlcyBlbHNlICctJzo+OH0gIHRzPXt0c30iKQogICAgaW5wdXQoJ1xuICBFbnRlci4uLicpCgoKZGVmIGNtZF9kZWxldGVfdHJhbnNmZXIoY29uZmlnKToKICAgIHNpZCA9IGVuc3VyZV9zaWQoY29uZmlnKQogICAgbWFuYWdlciA9IFRyYW5zZmVyaXRNYW5hZ2VyKHNpZCwgY29uZmlnKQogICAgdHJ5OgogICAgICAgIHRyYW5zZmVycyA9IG1hbmFnZXIubGlzdF90cmFuc2ZlcnMoKQogICAgZXhjZXB0IE1lZ2FBUElFcnJvciBhcyBleGM6CiAgICAgICAgcHJpbnQoZXIoZiIgIEdhZ2FsIG1lbmdhbWJpbCB0cmFuc2Zlcjoge2V4Y30iKSkKICAgICAgICByZXR1cm4KICAgIGlmIG5vdCB0cmFuc2ZlcnM6CiAgICAgICAgcHJpbnQoeWwoIiAgVGlkYWsgYWRhIHRyYW5zZmVyIHlhbmcgZGl0ZW11a2FuLiIpKQogICAgICAgIHJldHVybgogICAgY2hvaWNlcyA9IHtzdHIoaSk6IHQuZ2V0KCJ4aCIsICIiKSBmb3IgaSwgdCBpbiBlbnVtZXJhdGUodHJhbnNmZXJzLCAxKX0KICAgIGZvciBpLCB0IGluIGVudW1lcmF0ZSh0cmFuc2ZlcnMsIDEpOgogICAgICAgIHByaW50KGYiICBbe2l9XSAge3QuZ2V0KCd4aCcsICcnKX0iKQogICAgcGljayA9IGlucHV0KGN5KCIgIFBpbGloIG5vbW9yIHRyYW5zZmVyIHVudHVrIGRpaGFwdXM6ICIpKS5zdHJpcCgpCiAgICB4aCA9IGNob2ljZXMuZ2V0KHBpY2spCiAgICBpZiBub3QgeGg6CiAgICAgICAgcHJpbnQoZXIoIiAgUGlsaWhhbiB0aWRhayB2YWxpZC4iKSkKICAgICAgICByZXR1cm4KICAgIHkgPSBpbnB1dChlcihmIiAgWWFraW4gaGFwdXMgdHJhbnNmZXIge3hofT8gKHkvbik6ICIpKS5zdHJpcCgpLmxvd2VyKCkKICAgIGlmIHkgIT0gJ3knOgogICAgICAgIHByaW50KGRpbSgiICBEaWJhdGFsa2FuLiIpKQogICAgICAgIHJldHVybgogICAgdHJ5OgogICAgICAgIG1hbmFnZXIuZGVsZXRlX3RyYW5zZmVyKHhoKQogICAgICAgIHByaW50KG9rKGYiICBUcmFuc2ZlciB7eGh9IGJlcmhhc2lsIGRpaGFwdXMuIikpCiAgICBleGNlcHQgTWVnYUFQSUVycm9yIGFzIGV4YzoKICAgICAgICBwcmludChlcihmIiAgR2FnYWwgbWVuZ2hhcHVzOiB7ZXhjfSIpKQoKCmRlZiBjbWRfc2V0dGluZ3MoY29uZmlnKToKICAgIGhkcignUEVOR0FUVVJBTicpCiAgICBwcmludChmIiAgU0lEIHRlcnNpbXBhbiA6IHsnWWEnIGlmIGNvbmZpZy5nZXQoJ3NpZCcpIGVsc2UgJ0JlbHVtJ30iKQogICAgcHJpbnQoZiIgIEhhcmkgZGVmYXVsdCAgOiB7Y29uZmlnLmdldCgnZGF5cycpfSIpCiAgICBwcmludChmIiAgSmVkYSAgICAgICAgIDoge2NvbmZpZy5nZXQoJ2RlbGF5X21pbicpfSAtIHtjb25maWcuZ2V0KCdkZWxheV9tYXgnKX0gZGV0aWsiKQogICAgcHJpbnQoKQogICAgZGF5c19pbiA9IGlucHV0KGYiICBIYXJpIGFrdGlmIGRlZmF1bHQgW3tjb25maWcuZ2V0KCdkYXlzJywgOTApfV06ICIpLnN0cmlwKCkKICAgIGlmIGRheXNfaW4ubHN0cmlwKCctJykuaXNkaWdpdCgpOgogICAgICAgIGNvbmZpZ1siZGF5cyJdID0gaW50KGRheXNfaW4pCiAgICBkbWluID0gaW5wdXQoZiIgIEplZGEgbWluaW11bSAoZGV0aWspIFt7Y29uZmlnLmdldCgnZGVsYXlfbWluJywgMC42KX1dOiAiKS5zdHJpcCgpCiAgICBpZiBkbWluOgogICAgICAgIHRyeToKICAgICAgICAgICAgY29uZmlnWyJkZWxheV9taW4iXSA9IGZsb2F0KGRtaW4pCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgIGRtYXggPSBpbnB1dChmIiAgSmVkYSBtYWtzaW11bSAoZGV0aWspIFt7Y29uZmlnLmdldCgnZGVsYXlfbWF4JywgMS40KX1dOiAiKS5zdHJpcCgpCiAgICBpZiBkbWF4OgogICAgICAgIHRyeToKICAgICAgICAgICAgY29uZmlnWyJkZWxheV9tYXgiXSA9IGZsb2F0KGRtYXgpCiAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6CiAgICAgICAgICAgIHBhc3MKICAgIHIgPSBpbnB1dCgiICBSZXNldCBTSUQgKGhhcHVzIHlhbmcgdGVyc2ltcGFuKT8gKHkvbiwgRW50ZXI9bik6ICIpLnN0cmlwKCkubG93ZXIoKQogICAgaWYgciA9PSAneSc6CiAgICAgICAgY29uZmlnWyJzaWQiXSA9ICIiCiAgICBzYXZlX2NvbmZpZyhjb25maWcpCiAgICBwcmludChvaygiICBQZW5nYXR1cmFuIHRlcnNpbXBhbi4iKSkKCgpkZWYgbWFpbigpOgogICAgY29uZmlnID0gbG9hZF9jb25maWcoKQogICAgd2hpbGUgVHJ1ZToKICAgICAgICBjaSgpCiAgICAgICAgcHJpbnQoJ1xuJyArICc9JyAqIDYwKQogICAgICAgIHByaW50KGN5KCcgICBUUkFOU0ZFUi5JVCBBVVRPLVJFTkVXJykpCiAgICAgICAgcHJpbnQoZGltKCcgICBQZXJwYW5qYW5nICYgaGlkdXBrYW4ga2VtYmFsaSBzZW11YSB0cmFuc2ZlciBrZSA5MCBoYXJpJykpCiAgICAgICAgcHJpbnQoJz0nICogNjApCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KCcgIFsxXSBSZW5ldyAmIFJldml2ZSBTZW11YSBUcmFuc2ZlcicpCiAgICAgICAgcHJpbnQoJyAgWzJdIExpaGF0IFNlbXVhIFRyYW5zZmVyJykKICAgICAgICBwcmludCgnICBbM10gSGFwdXMgVHJhbnNmZXInKQogICAgICAgIHByaW50KCcgIFs0XSBQZW5nYXR1cmFuJykKICAgICAgICBwcmludCgnICBbNV0gS2VsdWFyJykKICAgICAgICBwcmludCgpCiAgICAgICAgYyA9IGlucHV0KCcgIFBpbGloIG1lbnU6ICcpLnN0cmlwKCkKICAgICAgICBpZiBjID09ICcxJzoKICAgICAgICAgICAgY21kX3JlbmV3X2FsbChjb25maWcpCiAgICAgICAgZWxpZiBjID09ICcyJzoKICAgICAgICAgICAgY21kX2xpc3RfdHJhbnNmZXJzKGNvbmZpZykKICAgICAgICBlbGlmIGMgPT0gJzMnOgogICAgICAgICAgICBjbWRfZGVsZXRlX3RyYW5zZmVyKGNvbmZpZykKICAgICAgICBlbGlmIGMgPT0gJzQnOgogICAgICAgICAgICBjbWRfc2V0dGluZ3MoY29uZmlnKQogICAgICAgIGVsaWYgYyA9PSAnNSc6CiAgICAgICAgICAgIHByaW50KCdcbiAgU2FtcGFpIGp1bXBhIScpCiAgICAgICAgICAgIHJldHVybgoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICB0cnk6CiAgICAgICAgbWFpbigpCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgcHJpbnQoJ1xuICBEaWJhdGFsa2FuLicp""",
        'haru-sub': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJoYXJ1LXN1YiAtIFN1YlNvdXJjZSBzdWJ0aXRsZSBkb3dubG9hZGVyLgpBUEk6IGh0dHBzOi8vYXBpLnN1YnNvdXJjZS5uZXQgICh3YWppYiBYLUFQSS1LZXkgZGFyaSBkYXNoYm9hcmQgcHJvZmlsZSBzdWJzb3VyY2UubmV0KQpLZXkgZGliYWNhIGRhcmkgc2VjcmV0IFNVQlNPVVJDRV9BUElfS0VZIC8gZW52IC8gL2NvbnRlbnQvLnN1YnNvdXJjZV9rZXkuCk1lbnlpbXBhbiBoYXNpbCBrZSAvY29udGVudC9kb3dubG9hZHMvc3VidGl0bGVzLCBvcHNpIGtpcmltIGtlIGJvdC4KIiIiCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHppcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcmVxdWVzdHMKCkFQSV9CQVNFID0gImh0dHBzOi8vYXBpLnN1YnNvdXJjZS5uZXQiCk9VVF9ESVIgPSBQYXRoKCIvY29udGVudC9kb3dubG9hZHMvc3VidGl0bGVzIikKS0VZX0ZJTEUgPSBQYXRoKCIvY29udGVudC8uc3Vic291cmNlX2tleSIpCgpMQU5HX01BUCA9IHsKICAgICJpZCI6ICJpbmRvbmVzaWFuIiwgImVuZyI6ICJlbmdsaXNoIiwgImVuIjogImVuZ2xpc2giLCAiZXMiOiAic3BhbmlzaCIsCiAgICAiZnIiOiAiZnJlbmNoIiwgImRlIjogImdlcm1hbiIsICJwdCI6ICJwb3J0dWd1ZXNlIiwgIml0IjogIml0YWxpYW4iLAogICAgIm5sIjogImR1dGNoIiwgInRyIjogInR1cmtpc2giLCAicnUiOiAicnVzc2lhbiIsICJhciI6ICJhcmFiaWMiLAogICAgImphIjogImphcGFuZXNlIiwgImtvIjogImtvcmVhbiIsICJ6aCI6ICJjaGluZXNlIiwgImhpIjogImhpbmRpIiwKICAgICJ2aSI6ICJ2aWV0bmFtZXNlIiwgInRoIjogInRoYWkiLCAibXMiOiAibWFsYXkiLCAicGwiOiAicG9saXNoIiwKfQoKU1VCX0VYVFMgPSAoIi5zcnQiLCAiLmFzcyIsICIuc3NhIiwgIi5zdWIiLCAiLnZ0dCIsICIuaWR4IikKCmRlZiBjaSgpOgogICAgc3lzLnN0ZG91dC53cml0ZSgnXHgxYlsySlx4MWJbSCcpCiAgICBzeXMuc3Rkb3V0LmZsdXNoKCkKCmRlZiBvayh0KTogcmV0dXJuICdcMDMzWzkybScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBlcih0KTogcmV0dXJuICdcMDMzWzkxbScgKyB0ICsgJ1wwMzNbMG0nCmRlZiBkaW0odCk6IHJldHVybiAnXDAzM1s5MG0nICsgdCArICdcMDMzWzBtJwpkZWYgY3kodCk6IHJldHVybiAnXDAzM1s5Nm0nICsgdCArICdcMDMzWzBtJwpkZWYgeWwodCk6IHJldHVybiAnXDAzM1s5M20nICsgdCArICdcMDMzWzBtJwoKZGVmIGhkcih0aXRsZSk6CiAgICBwcmludCgnXG4nICsgJz0nICogNjIpCiAgICBwcmludCgnICAnICsgdGl0bGUpCiAgICBwcmludCgnPScgKiA2MikKCiMgLS0tLS0tLS0tLSBzZWNyZXRzIC8gdGVsZWdyYW0gLS0tLS0tLS0tLQpkZWYgbG9hZF9zZWNyZXRzKCk6CiAgICB0cnk6CiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpOgogICAgICAgICAgICBkID0ganNvbi5sb2FkKG9wZW4oJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpKQogICAgICAgICAgICBmb3IgaywgdiBpbiBkLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBpZiB2IGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoayk6CiAgICAgICAgICAgICAgICAgICAgb3MuZW52aXJvbltrXSA9IHN0cih2KQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCgpkZWYgZ2V0X2FwaV9rZXkoKToKICAgIGxvYWRfc2VjcmV0cygpCiAgICB2ID0gb3MuZW52aXJvbi5nZXQoJ1NVQlNPVVJDRV9BUElfS0VZJywgJycpLnN0cmlwKCkKICAgIGlmIHY6CiAgICAgICAgcmV0dXJuIHYKICAgIGlmIEtFWV9GSUxFLmV4aXN0cygpOgogICAgICAgIGsgPSBLRVlfRklMRS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04Jykuc3RyaXAoKQogICAgICAgIGlmIGs6CiAgICAgICAgICAgIHJldHVybiBrCiAgICB0cnk6CiAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgdiA9IHN0cih1c2VyZGF0YS5nZXQoJ1NVQlNPVVJDRV9BUElfS0VZJykgb3IgJycpLnN0cmlwKCkKICAgICAgICBpZiB2OgogICAgICAgICAgICByZXR1cm4gdgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCiAgICByZXR1cm4gTm9uZQoKZGVmIHNhdmVfYXBpX2tleShrZXkpOgogICAgdHJ5OgogICAgICAgIEtFWV9GSUxFLndyaXRlX3RleHQoa2V5LnN0cmlwKCksIGVuY29kaW5nPSd1dGYtOCcpCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIEZhbHNlCgpkZWYgdGdfY3JlZGVudGlhbHMoKToKICAgIGxvYWRfc2VjcmV0cygpCiAgICB0b2sgPSBvcy5lbnZpcm9uLmdldCgnSEFSVV9CT1RfVE9LRU4nLCAnJykKICAgIG9pZCA9IG9zLmVudmlyb24uZ2V0KCdPV05FUl9JRCcsICcnKQogICAgaWYgbm90IHRvayBvciBub3Qgb2lkOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgICAgICAgICAgIGlmIG5vdCB0b2s6CiAgICAgICAgICAgICAgICB0b2sgPSBzdHIodXNlcmRhdGEuZ2V0KCdIQVJVX0JPVF9UT0tFTicpIG9yICcnKQogICAgICAgICAgICBpZiBub3Qgb2lkOgogICAgICAgICAgICAgICAgb2lkID0gc3RyKHVzZXJkYXRhLmdldCgnT1dORVJfSUQnKSBvciAnJykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCiAgICByZXR1cm4gdG9rLCBvaWQKCmRlZiB0Z19zZW5kX2RvY3VtZW50KHBhdGgsIGNhcHRpb249JycpOgogICAgdG9rLCBvaWQgPSB0Z19jcmVkZW50aWFscygpCiAgICBpZiBub3QgdG9rIG9yIG5vdCBvaWQgb3Igbm90IG9zLnBhdGguZXhpc3RzKHN0cihwYXRoKSk6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICB0cnk6CiAgICAgICAgd2l0aCBvcGVuKHN0cihwYXRoKSwgJ3JiJykgYXMgZmg6CiAgICAgICAgICAgIHIgPSByZXF1ZXN0cy5wb3N0KCdodHRwczovL2FwaS50ZWxlZ3JhbS5vcmcvYm90JyArIHRvayArICcvc2VuZERvY3VtZW50JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGF0YT17J2NoYXRfaWQnOiBvaWQsICdjYXB0aW9uJzogY2FwdGlvbn0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZpbGVzPXsnZG9jdW1lbnQnOiAoUGF0aChwYXRoKS5uYW1lLCBmaCl9LCB0aW1lb3V0PTYwKQogICAgICAgIHJldHVybiByLnN0YXR1c19jb2RlID09IDIwMAogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gRmFsc2UKCiMgLS0tLS0tLS0tLSBzdWJzb3VyY2UgYXBpIC0tLS0tLS0tLS0KZGVmIGFwaV9nZXQocGF0aCwgcGFyYW1zPU5vbmUsIHJldHJpZXM9Myk6CiAgICBrZXkgPSBnZXRfYXBpX2tleSgpCiAgICBoZWFkZXJzID0geyJYLUFQSS1LZXkiOiBrZXksICJBY2NlcHQiOiAiYXBwbGljYXRpb24vanNvbiJ9CiAgICB1cmwgPSBBUElfQkFTRSArICIvYXBpL3YxIiArIHBhdGgKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKHJldHJpZXMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgciA9IHJlcXVlc3RzLmdldCh1cmwsIHBhcmFtcz1wYXJhbXMgb3Ige30sIGhlYWRlcnM9aGVhZGVycywgdGltZW91dD0yMCkKICAgICAgICBleGNlcHQgcmVxdWVzdHMuZXhjZXB0aW9ucy5SZXF1ZXN0RXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHByaW50KGVyKGYiICBOZXR3b3JrIGVycm9yOiB7ZX0iKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiByLnN0YXR1c19jb2RlID09IDQyOToKICAgICAgICAgICAgd2FpdCA9IDUgKiAoYXR0ZW1wdCArIDEpCiAgICAgICAgICAgIHByaW50KHlsKGYiICDij7MgUmF0ZSBsaW1pdGVkLiBUdW5nZ3Uge3dhaXR9cy4uLiIpKQogICAgICAgICAgICB0aW1lLnNsZWVwKHdhaXQpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgci5zdGF0dXNfY29kZSA9PSA0MDE6CiAgICAgICAgICAgIHByaW50KGVyKCIgIEFQSSBrZXkgaW52YWxpZC9leHBpcmVkLiBDZWsga2V5IGRpIHN1YnNvdXJjZS5uZXQvZGFzaGJvYXJkL3Byb2ZpbGUiKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiByLnN0YXR1c19jb2RlICE9IDIwMDoKICAgICAgICAgICAgcHJpbnQoZXIoZiIgIEhUVFAge3Iuc3RhdHVzX2NvZGV9OiB7ci50ZXh0WzoxMjBdfSIpKQogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgcmV0dXJuIHIuanNvbigpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiBOb25lCgpkZWYgc2VhcmNoX21vdmllcyhxdWVyeSwgbXR5cGU9ImFsbCIsIHllYXI9Tm9uZSk6CiAgICBwYXJhbXMgPSB7InNlYXJjaFR5cGUiOiAidGV4dCIsICJxIjogcXVlcnl9CiAgICBpZiBtdHlwZSBhbmQgbXR5cGUgaW4gKCJtb3ZpZSIsICJzZXJpZXMiKToKICAgICAgICBwYXJhbXNbInR5cGUiXSA9IG10eXBlCiAgICBpZiB5ZWFyOgogICAgICAgIHBhcmFtc1sieWVhciJdID0geWVhcgogICAgZGF0YSA9IGFwaV9nZXQoIi9tb3ZpZXMvc2VhcmNoIiwgcGFyYW1zKQogICAgaWYgZGF0YSBhbmQgaXNpbnN0YW5jZShkYXRhLmdldCgiZGF0YSIpLCBsaXN0KToKICAgICAgICByZXR1cm4gZGF0YVsiZGF0YSJdCiAgICByZXR1cm4gW10KCmRlZiBnZXRfc3VidGl0bGVzKG1vdmllX2lkLCBsYW5ndWFnZT1Ob25lLCBsaW1pdD0zMCwgc29ydD0icG9wdWxhciIpOgogICAgcGFyYW1zID0geyJtb3ZpZUlkIjogbW92aWVfaWQsICJsaW1pdCI6IGxpbWl0LCAic29ydCI6IHNvcnR9CiAgICBpZiBsYW5ndWFnZToKICAgICAgICBwYXJhbXNbImxhbmd1YWdlIl0gPSBsYW5ndWFnZQogICAgZGF0YSA9IGFwaV9nZXQoIi9zdWJ0aXRsZXMiLCBwYXJhbXMpCiAgICBpZiBkYXRhIGFuZCBpc2luc3RhbmNlKGRhdGEuZ2V0KCJkYXRhIiksIGxpc3QpOgogICAgICAgIHJldHVybiBkYXRhWyJkYXRhIl0KICAgIHJldHVybiBbXQoKZGVmIGRvd25sb2FkX3N1YnRpdGxlKHN1YnRpdGxlX2lkKToKICAgIGtleSA9IGdldF9hcGlfa2V5KCkKICAgIGhlYWRlcnMgPSB7IlgtQVBJLUtleSI6IGtleSwgIkFjY2VwdCI6ICJhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0ifQogICAgdXJsID0gZiJ7QVBJX0JBU0V9L2FwaS92MS9zdWJ0aXRsZXMve3N1YnRpdGxlX2lkfS9kb3dubG9hZCIKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDMpOgogICAgICAgIHRyeToKICAgICAgICAgICAgciA9IHJlcXVlc3RzLmdldCh1cmwsIGhlYWRlcnM9aGVhZGVycywgdGltZW91dD0zMCkKICAgICAgICBleGNlcHQgcmVxdWVzdHMuZXhjZXB0aW9ucy5SZXF1ZXN0RXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIHByaW50KGVyKGYiICBOZXR3b3JrIGVycm9yOiB7ZX0iKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBpZiByLnN0YXR1c19jb2RlID09IDQyOToKICAgICAgICAgICAgdGltZS5zbGVlcCg1ICogKGF0dGVtcHQgKyAxKSkKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiByLnN0YXR1c19jb2RlID09IDIwMDoKICAgICAgICAgICAgcmV0dXJuIHIuY29udGVudAogICAgICAgIGlmIHIuc3RhdHVzX2NvZGUgPT0gNDAxOgogICAgICAgICAgICBwcmludChlcigiICBBUEkga2V5IGludmFsaWQvZXhwaXJlZC4iKSkKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBwcmludChlcihmIiAgSFRUUCB7ci5zdGF0dXNfY29kZX06IHtyLnRleHRbOjEyMF19IikpCiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiBOb25lCgpkZWYgZ2V0X3N1YnRpdGxlX2RldGFpbChzdWJ0aXRsZV9pZCk6CiAgICBkYXRhID0gYXBpX2dldChmIi9zdWJ0aXRsZXMve3N1YnRpdGxlX2lkfSIpCiAgICBpZiBkYXRhIGFuZCBpc2luc3RhbmNlKGRhdGEuZ2V0KCJkYXRhIiksIGRpY3QpOgogICAgICAgIHJldHVybiBkYXRhWyJkYXRhIl0KICAgIGlmIGRhdGEgYW5kIGlzaW5zdGFuY2UoZGF0YS5nZXQoImRhdGEiKSwgbGlzdCkgYW5kIGRhdGFbImRhdGEiXToKICAgICAgICByZXR1cm4gZGF0YVsiZGF0YSJdWzBdCiAgICByZXR1cm4gTm9uZQoKZGVmIHNhbml0aXplKG5hbWUpOgogICAgbmFtZSA9IHJlLnN1YihyJ1s8PjoiL1xcfD8qXHgwMC1ceDFmXScsICcnLCBuYW1lKQogICAgcmV0dXJuIHJlLnN1YihyJ1xzKycsICcgJywgbmFtZSkuc3RyaXAoKVs6MTIwXSBvciAndW50aXRsZWQnCgpkZWYgZXh0cmFjdF9zdWJ0aXRsZXMoemlwX2J5dGVzLCBvdXRfZGlyLCBiYXNlX25hbWUpOgogICAgc2F2ZWQgPSBbXQogICAgdHJ5OgogICAgICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGlvLkJ5dGVzSU8oemlwX2J5dGVzKSkgYXMgejoKICAgICAgICAgICAgbmFtZXMgPSBbbiBmb3IgbiBpbiB6Lm5hbWVsaXN0KCkKICAgICAgICAgICAgICAgICAgICAgaWYgbm90IG4uZW5kc3dpdGgoIi8iKSBhbmQgbi5sb3dlcigpLmVuZHN3aXRoKFNVQl9FWFRTKV0KICAgICAgICAgICAgIyBwcmVmZXIgc3J0IHBlcnRhbWEKICAgICAgICAgICAgbmFtZXMuc29ydChrZXk9bGFtYmRhIG46IChuLmxvd2VyKCkuZW5kc3dpdGgoIi5zcnQiKSwgbikpCiAgICAgICAgICAgIGZvciBpLCBuIGluIGVudW1lcmF0ZShuYW1lcyk6CiAgICAgICAgICAgICAgICBleHQgPSBQYXRoKG4pLnN1ZmZpeAogICAgICAgICAgICAgICAgZGF0YSA9IHoucmVhZChuKQogICAgICAgICAgICAgICAgaWYgbGVuKG5hbWVzKSA9PSAxOgogICAgICAgICAgICAgICAgICAgIGZuYW1lID0gZiJ7YmFzZV9uYW1lfXtleHR9IgogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICBmbmFtZSA9IGYie2Jhc2VfbmFtZX1fe2krMX17ZXh0fSIKICAgICAgICAgICAgICAgIG91dCA9IG91dF9kaXIgLyBmbmFtZQogICAgICAgICAgICAgICAgb3V0LndyaXRlX2J5dGVzKGRhdGEpCiAgICAgICAgICAgICAgICBzYXZlZC5hcHBlbmQob3V0KQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGVyKGYiICBHYWdhbCBleHRyYWN0OiB7ZX0iKSkKICAgIHJldHVybiBzYXZlZAoKZGVmIGh1bWFuX3NpemUobnVtKToKICAgIGZvciB1bml0IGluICgiQiIsICJLQiIsICJNQiIsICJHQiIpOgogICAgICAgIGlmIGFicyhudW0pIDwgMTAyNC4wOgogICAgICAgICAgICByZXR1cm4gZiJ7bnVtIGlmIHVuaXQgPT0gJ0InIGVsc2UgZid7bnVtOi4xZn0nfSB7dW5pdH0iCiAgICAgICAgbnVtIC89IDEwMjQuMAogICAgcmV0dXJuIGYie251bTouMWZ9IFRCIgoKIyAtLS0tLS0tLS0tIGludGVyYWN0aXZlIC0tLS0tLS0tLS0KZGVmIGFza19rZXkoKToKICAgIGtleSA9IGdldF9hcGlfa2V5KCkKICAgIGlmIGtleToKICAgICAgICByZXR1cm4ga2V5CiAgICBwcmludCh5bCgiXG4gIPCflJEgQVBJIEtleSBTdWJTb3VyY2UgYmVsdW0gZGlrb25maWd1cmFzaS4iKSkKICAgIHByaW50KGRpbSgiICBDYXJhIGRhcGF0OlxuICAgIDEuIEJ1a2EgaHR0cHM6Ly9zdWJzb3VyY2UubmV0XG4gICAgMi4gTG9naW4gYXRhdSBidWF0IGFrdW5cbiAgICAzLiBNZW51IFByb2ZpbGUg4oaSIEFQSSBLZXlcbiAgICA0LiBTYWxpbiBrZXktbnlhIikpCiAgICBrZXkgPSBpbnB1dChjeSgiICBQYXN0ZSBBUEkgS2V5OiAiKSkuc3RyaXAoKQogICAgd2hpbGUgbm90IGtleToKICAgICAgICBwcmludChlcigiICBBUEkgS2V5IHRpZGFrIGJvbGVoIGtvc29uZy4iKSkKICAgICAgICBrZXkgPSBpbnB1dChjeSgiICBQYXN0ZSBBUEkgS2V5OiAiKSkuc3RyaXAoKQogICAgb3MuZW52aXJvblsnU1VCU09VUkNFX0FQSV9LRVknXSA9IGtleQogICAgc2F2ZV9hcGlfa2V5KGtleSkKICAgIHByaW50KG9rKCIgIEFQSSBLZXkgdGVyc2ltcGFuIChndW5ha2FuIHNlY3JldCBTVUJTT1VSQ0VfQVBJX0tFWSBiaWFyIHBlcm1hbmVuKS4iKSkKICAgIHJldHVybiBrZXkKCmRlZiBtYWluKCk6CiAgICBpZiBub3QgZ2V0X2FwaV9rZXkoKToKICAgICAgICBhc2tfa2V5KCkKICAgIHdoaWxlIFRydWU6CiAgICAgICAgY2koKQogICAgICAgIHByaW50KCdcbicgKyAnPScgKiA2MikKICAgICAgICBwcmludChjeSgnICAgaGFydS1zdWIgLS0gU3ViU291cmNlIFN1YnRpdGxlIERvd25sb2FkZXInKSkKICAgICAgICBwcmludChkaW0oJyAgIEFQSTogc3Vic291cmNlLm5ldCB8IFgtQVBJLUtleScpKQogICAgICAgIHByaW50KCc9JyAqIDYyKQogICAgICAgIHByaW50KCkKICAgICAgICBwcmludCgnICBDYXJpIHN1YnRpdGxlIGp1ZHVsIGZpbG0vc2VyaS4gQ29udG9oIHF1ZXJ5OicpCiAgICAgICAgcHJpbnQoJyAgICAiSW5jZXB0aW9uIiAgLyAgIlRoZSBCZWFyIDIwMjIiICAvICAiTmFydXRvIDIwMDIiJykKICAgICAgICBwcmludCgpCiAgICAgICAgcSA9IGlucHV0KCcgIEp1ZHVsIChlbnRlcj1rZWx1YXIpOiAnKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHE6CiAgICAgICAgICAgIHByaW50KCdcbiAgQnllIScpCiAgICAgICAgICAgIHJldHVybgogICAgICAgIG10eXBlID0gaW5wdXQoJyAgVGlwZSBbbW92aWUvc2VyaWVzL2FsbF0gKEVudGVyPWFsbCk6ICcpLnN0cmlwKCkubG93ZXIoKSBvciAnYWxsJwogICAgICAgIHllYXIgPSBOb25lCiAgICAgICAgeW0gPSByZS5zZWFyY2gocidcYigxOXwyMClcZHsyfVxiJywgcSkKICAgICAgICBpZiB5bToKICAgICAgICAgICAgeWVhciA9IHltLmdyb3VwKDApCiAgICAgICAgICAgIHEgPSBxLnJlcGxhY2UoeWVhciwgJycpLnN0cmlwKCcgLScpCiAgICAgICAgcHJpbnQoZidcbiAgTWVuY2FyaSAie3F9IicgKyAoZicgKHt5ZWFyfSknIGlmIHllYXIgZWxzZSAnJykgKyAnLi4uJykKICAgICAgICByZXN1bHRzID0gc2VhcmNoX21vdmllcyhxLCBtdHlwZSwgeWVhcikKICAgICAgICBpZiBub3QgcmVzdWx0czoKICAgICAgICAgICAgcHJpbnQoeWwoJyAgVGlkYWsgYWRhIGhhc2lsLicpKQogICAgICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmludChmJ1xuICBEaXRlbXVrYW4ge2xlbihyZXN1bHRzKX0ganVkdWw6JykKICAgICAgICBmb3IgaSwgbSBpbiBlbnVtZXJhdGUocmVzdWx0cywgMSk6CiAgICAgICAgICAgIHN1YnMgPSBtLmdldCgnc3VidGl0bGVDb3VudCcpCiAgICAgICAgICAgIHN1YnNfcyA9IGYiICh7c3Vic30gc3ViKSIgaWYgc3VicyBpcyBub3QgTm9uZSBlbHNlICIiCiAgICAgICAgICAgIHByaW50KGYiICBbe2l9XSB7bS5nZXQoJ3RpdGxlJywnPycpfSAoe20uZ2V0KCdyZWxlYXNlWWVhcicsJz8nKX0pIFt7bS5nZXQoJ3R5cGUnLCc/Jyl9XXtzdWJzX3N9IikKICAgICAgICBzZWwgPSBpbnB1dChjeShmIlxuICBQaWxpaCBub21vciBbMS17bGVuKHJlc3VsdHMpfV06ICIpKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHNlbC5sc3RyaXAoJy0nKS5pc2RpZ2l0KCkgb3Igbm90ICgxIDw9IGludChzZWwpIDw9IGxlbihyZXN1bHRzKSk6CiAgICAgICAgICAgIHByaW50KGVyKCcgIFBpbGloYW4gdGlkYWsgdmFsaWQuJykpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbW92aWUgPSByZXN1bHRzW2ludChzZWwpIC0gMV0KICAgICAgICBtaWQgPSBtb3ZpZS5nZXQoJ21vdmllSWQnKQogICAgICAgIHRpdGxlID0gc2FuaXRpemUobW92aWUuZ2V0KCd0aXRsZScpIG9yICdUaXRsZScpCiAgICAgICAgcHJpbnQoZiJcbiAg8J+TiyBKdWR1bDoge21vdmllLmdldCgndGl0bGUnKX0gKHttb3ZpZS5nZXQoJ3JlbGVhc2VZZWFyJyl9KSIpCiAgICAgICAgbGFuZ19xID0gaW5wdXQoJyAgQmFoYXNhIChrb3Nvbmc9c2VtdWEsIGNvbnRvaDogaWQvZW5nbGlzaC9pbmRvbmVzaWFuKSBbaWRdOiAnKS5zdHJpcCgpLmxvd2VyKCkgb3IgJ2lkJwogICAgICAgIGxhbmdfbmFtZSA9IE5vbmUKICAgICAgICBpZiBsYW5nX3E6CiAgICAgICAgICAgIGxhbmdfbmFtZSA9IExBTkdfTUFQLmdldChsYW5nX3EsIGxhbmdfcSBpZiBsZW4obGFuZ19xKSA+IDMgZWxzZSBMQU5HX01BUC5nZXQobGFuZ19xLCBOb25lKSkKICAgICAgICBwcmludCgnICBNZW5nYW1iaWwgZGFmdGFyIHN1YnRpdGxlLi4uJykKICAgICAgICBzdWJzID0gZ2V0X3N1YnRpdGxlcyhtaWQsIGxhbmdfbmFtZSkKICAgICAgICBpZiBub3Qgc3ViczoKICAgICAgICAgICAgcHJpbnQoeWwoJyAgVGlkYWsgYWRhIHN1YnRpdGxlIHRlcnNlZGlhLicpKQogICAgICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKICAgICAgICAgICAgY29udGludWUKICAgICAgICB1cF9pZHMgPSBzb3J0ZWQoe3N0cihzLmdldCgndXBsb2FkZXJJZCcpIG9yICc/JykgZm9yIHMgaW4gc3Vic30pCiAgICAgICAgcHJpbnQoZidcbiAgU3VidGl0bGUgdGVyc2VkaWEgKHtsZW4oc3Vicyl9IHN1Yiwge2xlbih1cF9pZHMpfSB1cGxvYWRlcik6JykKICAgICAgICBmb3IgaSwgcyBpbiBlbnVtZXJhdGUoc3VicywgMSk6CiAgICAgICAgICAgIHJlbCA9ICIsICIuam9pbihzLmdldCgncmVsZWFzZUluZm8nKSBvciBbXSlbOjM4XSBvciAnLScKICAgICAgICAgICAgZGwgPSBzLmdldCgnZG93bmxvYWRzJykgb3IgMAogICAgICAgICAgICByYXRlID0gKHMuZ2V0KCdyYXRpbmcnKSBvciB7fSkuZ2V0KCdnb29kJykKICAgICAgICAgICAgcmF0ZV9zID0gZiLirZB7cmF0ZX0iIGlmIHJhdGUgZWxzZSAnJwogICAgICAgICAgICBoaSA9ICdISScgaWYgcy5nZXQoJ2hlYXJpbmdJbXBhaXJlZCcpIGVsc2UgJ05vJwogICAgICAgICAgICBzeiA9IGh1bWFuX3NpemUocy5nZXQoJ3NpemUnKSBvciAwKSBpZiBzLmdldCgnc2l6ZScpIGVsc2UgJycKICAgICAgICAgICAgbGFuZyA9IHN0cihzLmdldCgnbGFuZ3VhZ2UnKSBvciAnJykKICAgICAgICAgICAgdXBkID0gZiJVOntzLmdldCgndXBsb2FkZXJJZCcpfSIgaWYgcy5nZXQoJ3VwbG9hZGVySWQnKSBlbHNlICcnCiAgICAgICAgICAgIHByaW50KGYiICBbe2l9XSB7cmVsOjwzOH0ge2xhbmc6PDEwfSB7c3o6Pjd9IERMOntkbDo8NX0ge3JhdGVfczo8NH0gSEk6e2hpfSB7dXBkfSIpCiAgICAgICAgICAgIGNtdCA9IChzLmdldCgnY29tbWVudGFyeScpIG9yICcnKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIGNtdDoKICAgICAgICAgICAgICAgIGNtdDEgPSBjbXQucmVwbGFjZSgnXG4nLCAnICcpLnN0cmlwKClbOjYwXQogICAgICAgICAgICAgICAgcHJpbnQoZiIgICAgICAgIPCfkqwge2RpbShjbXQxKX0iKQogICAgICAgIHNlbDIgPSBpbnB1dChjeShmIlxuICBQaWxpaCBub21vciBzdWJ0aXRsZSBbMS17bGVuKHN1YnMpfV06ICIpKS5zdHJpcCgpCiAgICAgICAgaWYgbm90IHNlbDIubHN0cmlwKCctJykuaXNkaWdpdCgpIG9yIG5vdCAoMSA8PSBpbnQoc2VsMikgPD0gbGVuKHN1YnMpKToKICAgICAgICAgICAgcHJpbnQoZXIoJyAgUGlsaWhhbiB0aWRhayB2YWxpZC4nKSkKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzdWIgPSBzdWJzW2ludChzZWwyKSAtIDFdCiAgICAgICAgc2lkID0gc3ViLmdldCgnc3VidGl0bGVJZCcpCiAgICAgICAgZGV0YWlsID0gZ2V0X3N1YnRpdGxlX2RldGFpbChzaWQpIG9yIHN1YgogICAgICAgIGNtdCA9IChkZXRhaWwuZ2V0KCdjb21tZW50YXJ5Jykgb3IgJycpLnN0cmlwKCkKICAgICAgICBwcmV2ID0gKGRldGFpbC5nZXQoJ3ByZXZpZXcnKSBvciAnJykuc3RyaXAoKQogICAgICAgIHByaW50KCdcbicgKyAnLScgKiA2MikKICAgICAgICBwcmludChjeSgiICBEZXRhaWwgc3VidGl0bGUgIyVzIiAlIHNpZCkpCiAgICAgICAgcHJpbnQoJy0nICogNjIpCiAgICAgICAgcHJpbnQoZiIgIFJlbGVhc2UgICA6IHsnLCAnLmpvaW4oZGV0YWlsLmdldCgncmVsZWFzZUluZm8nKSBvciBbXSl9IikKICAgICAgICBwcmludChmIiAgTGFuZ3VhZ2UgIDoge2RldGFpbC5nZXQoJ2xhbmd1YWdlJyl9IikKICAgICAgICBwcmludChmIiAgVXBsb2FkZXIgIDoge2RldGFpbC5nZXQoJ3VwbG9hZGVySWQnKX0iKQogICAgICAgIHByaW50KGYiICBGcmFtZXJhdGUgOiB7ZGV0YWlsLmdldCgnZnJhbWVyYXRlJykgb3IgJy0nfSIpCiAgICAgICAgcHJpbnQoZiIgIFRpcGUgICAgICA6IHsoZGV0YWlsLmdldCgncHJvZHVjdGlvblR5cGUnKSBvciAnJyl9L3tkZXRhaWwuZ2V0KCdyZWxlYXNlVHlwZScpIG9yICctJ30iKQogICAgICAgIHByaW50KGYiICBEb3dubG9hZHMgOiB7ZGV0YWlsLmdldCgnZG93bmxvYWRzJykgb3IgMH0gICBVa3VyYW46IHtodW1hbl9zaXplKGRldGFpbC5nZXQoJ3NpemUnKSBvciAwKSBpZiBkZXRhaWwuZ2V0KCdzaXplJykgZWxzZSAnLSd9IikKICAgICAgICBpZiBjbXQ6CiAgICAgICAgICAgIHByaW50KGYiXG4gIPCfkqwge2NtdH0iKQogICAgICAgIGlmIHByZXY6CiAgICAgICAgICAgIHByZXZfbGluZXMgPSBbbCBmb3IgbCBpbiBwcmV2LnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCldWzoxMl0KICAgICAgICAgICAgcHJpbnQoJ1xuICDwn5GBIFByZXZpZXc6JykKICAgICAgICAgICAgZm9yIGwgaW4gcHJldl9saW5lczoKICAgICAgICAgICAgICAgIHByaW50KGRpbSgnICAgIHwgJyArIGwpKQogICAgICAgIHByaW50KCctJyAqIDYyKQogICAgICAgIHAgPSBpbnB1dCh5bCgnICBMYW5qdXQgZG93bmxvYWQgc3VidGl0bGUgaW5pPyBbWS9uXTogJykpLnN0cmlwKCkubG93ZXIoKQogICAgICAgIGlmIHAgbm90IGluICgneScsICd5ZXMnLCAnJyk6CiAgICAgICAgICAgIHByaW50KGRpbSgnICBEaWJhdGFsa2FuLicpKQogICAgICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmludChmIiAg4qyHIERvd25sb2FkIHN1YnRpdGxlICN7c2lkfS4uLiIpCiAgICAgICAgZGF0YSA9IGRvd25sb2FkX3N1YnRpdGxlKHNpZCkKICAgICAgICBpZiBub3QgZGF0YToKICAgICAgICAgICAgaW5wdXQoJ1xuICBFbnRlci4uLicpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgb3V0X2RpciA9IE9VVF9ESVIKICAgICAgICBvdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBiYXNlID0gZiJ7dGl0bGV9Lnttb3ZpZS5nZXQoJ3JlbGVhc2VZZWFyJywnJyl9Ii5zdHJpcCgnLicpCiAgICAgICAgaWYgbW92aWUuZ2V0KCd0eXBlJykgPT0gJ3NlcmllcycgYW5kIG1vdmllLmdldCgnc2Vhc29uJyk6CiAgICAgICAgICAgIGJhc2UgKz0gZiIuU3ttb3ZpZS5nZXQoJ3NlYXNvbicpfSIKICAgICAgICBzYXZlZCA9IGV4dHJhY3Rfc3VidGl0bGVzKGRhdGEsIG91dF9kaXIsIGJhc2UpCiAgICAgICAgaWYgc2F2ZWQ6CiAgICAgICAgICAgIHByaW50KG9rKGYiXG4gIOKchSBUZXJzaW1wYW46IHtsZW4oc2F2ZWQpfSBmaWxlIC0+IHtvdXRfZGlyfSIpKQogICAgICAgICAgICBmb3IgcyBpbiBzYXZlZDoKICAgICAgICAgICAgICAgIHByaW50KGRpbSgnICAgIOKAoiAnICsgcy5uYW1lKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIGdhZ2FsIHVuemlwIC0+IHNpbXBhbiBtZW50YWgKICAgICAgICAgICAgcmF3X3BhdGggPSBvdXRfZGlyIC8gZiJ7YmFzZX0uemlwIgogICAgICAgICAgICByYXdfcGF0aC53cml0ZV9ieXRlcyhkYXRhKQogICAgICAgICAgICBwcmludChvayhmIiAgRmlsZSB6aXAgdGVyc2ltcGFuOiB7cmF3X3BhdGh9IikpCiAgICAgICAgdG9rLCBvaWQgPSB0Z19jcmVkZW50aWFscygpCiAgICAgICAgaWYgc2F2ZWQgYW5kIHRvayBhbmQgb2lkOgogICAgICAgICAgICBwID0gaW5wdXQoeWwoJyAgS2lyaW0gc3VidGl0bGUga2UgYm90IFRlbGVncmFtPyBbeS9OXTogJykpLnN0cmlwKCkubG93ZXIoKQogICAgICAgICAgICBpZiBwIGluICgneScsICd5ZXMnKToKICAgICAgICAgICAgICAgIGZvciBzIGluIHNhdmVkOgogICAgICAgICAgICAgICAgICAgIHRnX3NlbmRfZG9jdW1lbnQocywgZiJoYXJ1LXN1Ylxue2Jhc2V9IikKICAgICAgICBpbnB1dCgnXG4gIEVudGVyLi4uJykKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgdHJ5OgogICAgICAgIG1haW4oKQogICAgZXhjZXB0IEtleWJvYXJkSW50ZXJydXB0OgogICAgICAgIHByaW50KCdcbiAgRGliYXRhbGthbi4nKQ==""",
    }
    for _name, _blob in TOOLS.items():
        _p = '/usr/local/bin/' + _name
        _code = base64.b64decode(_blob).decode('utf-8').replace('\r\n', '\n').replace('\r', '\n')
        if not _code.startswith('#!'):
            _code = '#!/usr/bin/env python3\n' + _code
        with open(_p, 'w', encoding='utf-8') as _f:
            _f.write(_code)
        os.chmod(_p, 0o755)
        print('  OK ' + _name)
    try:
        with open(os.path.expanduser('~/.bashrc'), 'a') as _bf:
            _bf.write("\nalias haru-lrc='/usr/local/bin/haru-lrc'\n")
            _bf.write("\nalias haru-manga='/usr/local/bin/haru-manga'\n")
            _bf.write("\nalias haru-ytdl='/usr/local/bin/haru-ytdl'\n")
            _bf.write("\nalias haru-check='/usr/local/bin/haru-check'\n")
            _bf.write("\nalias haru-transferit='/usr/local/bin/haru-transferit'\n")
            _bf.write("\nalias haru-sub='/usr/local/bin/haru-sub'\n")
    except Exception:
        pass
    for _d in ['/content/downloads/Video', '/content/downloads/Audio', '/content/downloads/Playlist', '/content/downloads/lrc', '/content/downloads/manga', '/content/downloads/subtitles', '/content/downloads/checker']:
        os.makedirs(_d, exist_ok=True)
    try:
        _secrets = {}
        try:
            from google.colab import userdata as _ud
            for _k in ['GOFILE_API_TOKEN','GDRIVE_CLIENT_ID','GDRIVE_CLIENT_SECRET','GDRIVE_REFRESH_TOKEN','GDRIVE_FOLDER_ID','OWNER_ID','HARU_BOT_TOKEN','SUBSOURCE_API_KEY']:
                try:
                    _v = _ud.get(_k)
                    if _v: _secrets[_k]=str(_v).strip()
                except Exception: pass
        except Exception: pass
        for _k,_v in _secrets.items(): os.environ[_k]=_v
        if _secrets:
            import json as _js
            _old = {}
            if os.path.exists('/content/.haru_secrets.json'):
                try: _old = _js.load(open('/content/.haru_secrets.json'))
                except Exception: pass
            _old.update(_secrets)
            with open('/content/.haru_secrets.json','w') as _sf: _js.dump(_old,_sf)
            os.chmod('/content/.haru_secrets.json',0o600)
            print('  Secrets untuk terminal: '+', '.join(sorted(_old.keys())))
        else:
            print('  (Belum ada secret terbaca - aktifkan di menu Rahasia.)')
    except Exception:
        print('  (Skip export secrets.)')
    print()
    print('Ketik di terminal: haru-ytdl | haru-lrc | haru-manga | haru-check | haru-transferit | haru-sub')
else:
    print('Install dinonaktifkan.')


## 2 — Web Terminal di Browser (ttyd + Cloudflare) — disarankan
Jalankan cell di bawah, klik link yang muncul. Copy-paste & arrow keys jalan.


In [ ]:
#@title Buka Web Terminal { display-mode: "form" }
import os, time, re, subprocess, requests
from IPython.display import HTML, display

print('Setup web terminal...')
try:
    _s2 = {}
    try:
        from google.colab import userdata as _ud
        for _k in ['GOFILE_API_TOKEN','GDRIVE_CLIENT_ID','GDRIVE_CLIENT_SECRET','GDRIVE_REFRESH_TOKEN','GDRIVE_FOLDER_ID','OWNER_ID','HARU_BOT_TOKEN']:
            try:
                _v = _ud.get(_k)
                if _v: _s2[_k]=str(_v).strip()
            except Exception: pass
    except Exception: pass
    if _s2:
        import json as _js
        try: _old = _js.load(open('/content/.haru_secrets.json'))
        except Exception: _old = {}
        _old.update(_s2)
        with open('/content/.haru_secrets.json','w') as _sf: _js.dump(_old,_sf)
        os.chmod('/content/.haru_secrets.json',0o600)
        print('  Secrets refresh: '+', '.join(sorted(_old.keys())))
except Exception: pass
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('  Download cloudflared...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', '/usr/local/bin/cloudflared'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])
if not os.path.exists('/usr/local/bin/ttyd'):
    print('  Download ttyd...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64', '-o', '/usr/local/bin/ttyd'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/ttyd'])
subprocess.run(['pkill', '-f', 'ttyd'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(1)
subprocess.run(['tmux', 'set', '-g', 'history-limit', '50000'], capture_output=True)
subprocess.run(['tmux', 'set', '-g', 'mouse', 'on'], capture_output=True)
subprocess.Popen(['/usr/local/bin/ttyd', '-p', '7681', '-W', '-t', 'fontSize=15', 'tmux', 'new-session', '-A', '-s', 'aio', 'bash'], cwd='/content', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
print('  Buka tunnel Cloudflare...')
cf = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7681'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
web_url = None
end = time.time() + 35
while time.time() < end:
    line = cf.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        web_url = m[-1]
        break
print()
print('=' * 62)
if web_url:
    print('WEB TERMINAL SIAP:')
    print('  ' + web_url)
    print()
    print('  Perintah: haru-ytdl | haru-lrc | haru-manga')
    display(HTML('<a href="' + web_url + '" target="_blank" style="background:#238636;color:#fff;padding:12px 24px;text-decoration:none;border-radius:6px;font-weight:bold;display:inline-block;">Buka Web Terminal</a>'))
    try:
        from google.colab import userdata as _ud
        _oid = _ud.get('OWNER_ID') or ''
    except Exception:
        _oid = ''
    try:
        from google.colab import userdata as _ud2
        _tg = _ud2.get('HARU_BOT_TOKEN') or ''
    except Exception:
        _tg = ''
    if _oid and _tg:
        try:
            requests.post('https://api.telegram.org/bot' + _tg + '/sendMessage', json={'chat_id': _oid, 'text': '<b>Haru AIO terminal siap!</b>\nWeb: ' + web_url + '\nKetik: haru-ytdl / haru-lrc / haru-manga', 'parse_mode': 'HTML', 'disable_web_page_preview': True}, timeout=8)
            print('  Notif Telegram terkirim.')
        except Exception as _e:
            print('  Gagal kirim Telegram.')
    else:
        print('  (Aktifkan HARU_BOT_TOKEN & OWNER_ID di Secrets biar link auto-post.)')
else:
    print('Gagal dapat URL tunnel. Jalankan ulang cell ini.')
print('=' * 62)
print('Biarkan cell ini running agar tunnel tetap hidup.')
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print('Web terminal ditutup.')


## 3 — Upload cookies.txt (opsional, untuk ytdl login)


In [ ]:
#@title Upload cookies.txt { display-mode: "form" }
try:
    from google.colab import files
    print('Pilih file cookies.txt dari PC:')
    up = files.upload()
    for name, data in up.items():
        dest = '/content/cookies.txt'
        open(dest, 'wb').write(data)
        print('OK tersimpan di /content/cookies.txt (%d bytes)' % len(data))
except ImportError:
    print('Bukan di Colab - skip.')

## 4 — Upload hasil


In [ ]:
#@title Upload hasil download { display-mode: "form" }
upload_path = "/content/downloads" #@param {type:"string"}
upload_target = "Gofile" #@param ["Gofile", "Google Drive"]

import subprocess, os, re, json, time, requests
from pathlib import Path

def _sec(k):
    v = os.environ.get(k, '')
    if v: return v.strip()
    try:
        from google.colab import userdata
        t = userdata.get(k)
        if t: return str(t).strip()
    except Exception: pass
    if os.path.exists('/content/.haru_secrets.json'):
        try:
            d = json.load(open('/content/.haru_secrets.json'))
            if d.get(k): return str(d[k]).strip()
        except Exception: pass
    return ''

def _tg(msg):
    tok, oid = _sec('HARU_BOT_TOKEN'), _sec('OWNER_ID')
    if not tok or not oid: return
    try: requests.post('https://api.telegram.org/bot'+tok+'/sendMessage', json={'chat_id': oid, 'text': msg, 'parse_mode': 'HTML', 'disable_web_page_preview': True}, timeout=10)
    except Exception: pass

p = Path(upload_path)
files = sorted([f for f in p.rglob('*') if f.is_file()]) if p.exists() else []
if not files:
    print('Tidak ada file di ' + str(p))
else:
    print('%d file:' % len(files))
    for i, f in enumerate(files): print('  [%d] %s (%.1f MB)' % (i, f.name, f.stat().st_size/1024/1024))
    sel = input('Pilih (nomor / * semua): ').strip()
    tgts = files if sel == '*' else [files[int(sel)]] if sel.isdigit() and 0 <= int(sel) < len(files) else []
    for f in tgts:
        if upload_target == 'Gofile':
            srv = 'store1'
            try:
                sv = requests.get('https://api.gofile.io/servers', timeout=15).json()
                if sv.get('status') == 'ok': srv = sv['data']['servers'][0]['name']
            except Exception: pass
            print('Upload %s via %s...' % (f.name, srv))
            r = subprocess.run(['curl','-s','-F','file=@'+str(f),'https://'+srv+'.gofile.io/uploadFile'], capture_output=True, text=True, timeout=600)
            try:
                d = json.loads(r.stdout)
                if d.get('status') == 'ok':
                    print('OK ' + d['data']['downloadPage'])
                    _tg('<b>Upload Gofile</b>\n'+f.name+'\n'+d['data']['downloadPage'])
                else: print('Gagal: ' + r.stdout[:200])
            except Exception: print('Gagal: ' + r.stdout[:200])
        else:
            cid, sec, ref = _sec('GDRIVE_CLIENT_ID'), _sec('GDRIVE_CLIENT_SECRET'), _sec('GDRIVE_REFRESH_TOKEN')
            folder = _sec('GDRIVE_FOLDER_ID') or 'HaruDownloads'
            if not (cid and sec and ref):
                print('Secret GDrive belum lengkap.')
                continue
            tok = requests.post('https://oauth2.googleapis.com/token', data={'client_id': cid, 'client_secret': sec, 'refresh_token': ref, 'grant_type': 'refresh_token'}, timeout=15).json().get('access_token')
            if not tok:
                print('Gagal auth GDrive.')
                continue
            q = "name='"+folder+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
            fl = requests.get('https://www.googleapis.com/drive/v3/files', headers={'Authorization': 'Bearer '+tok}, params={'q': q, 'fields': 'files(id,name)'}, timeout=15).json().get('files', [])
            if fl: fid = fl[0]['id']
            else: fid = requests.post('https://www.googleapis.com/drive/v3/files', headers={'Authorization': 'Bearer '+tok, 'Content-Type': 'application/json'}, data=json.dumps({'name': folder, 'mimeType': 'application/vnd.google-apps.folder'}), timeout=15).json().get('id')
            size = f.stat().st_size
            ri = requests.post('https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable', headers={'Authorization': 'Bearer '+tok, 'Content-Type': 'application/json', 'X-Upload-Content-Type': 'application/octet-stream', 'X-Upload-Content-Length': str(size)}, data=json.dumps({'name': f.name, 'parents': [fid]}), timeout=30)
            uri = ri.headers.get('Location')
            up, CH, t0, done = 0, 64*1024*1024 if size > 100*1024*1024 else 16*1024*1024, time.time(), False
            fh = open(f, 'rb')
            while up < size:
                ch = fh.read(CH)
                if not ch: break
                end = up + len(ch) - 1
                pr = requests.put(uri, headers={'Content-Range': 'bytes %d-%d/%d' % (up, end, size), 'Content-Length': str(len(ch))}, data=ch, timeout=120)
                if pr.status_code in (200, 201): up += len(ch); done = True; break
                elif pr.status_code == 308:
                    up += len(ch)
                    print('  %.1f%%' % (up/size*100))
                else: print('  HTTP %d' % pr.status_code); break
            fh.close()
            if done or up >= size:
                print('OK terupload ke GDrive.')
                _tg('<b>Upload GDrive</b>\n'+f.name)
